# Master Prompt and Publication Naming Guide

## Master Prompt for Copilot/AI
**Role**: Expert Data Scientist and Academic Editor.  
**Context**: I am preparing figures and tables for an Operations Management journal paper.

### Global Rules for Visualization and Tables
1. Apply naming maps silently to data values for the following columns, only when columns exist.
2. Remove plot-type words from titles (for example, Boxplot, Histogram).
3. Use the same compact Matplotlib-only styling as experiments.ipynb.
4. Keep this mapping guide in the notebook for reproducibility and reviewer clarity.

## Naming Maps Used in This Notebook

### ml_method_type
- ML_Augmented_BD -> LABD
- ML_Augmented_BD_with_duals -> LABD-D
- Uncertainty_Similarity -> uncrrtn
- Random_Sampling -> RND

### prediction_method
- KNN -> kNN
- TREE -> DT
- CLUSTERING -> CLU
- When present, this field is treated as the reporting label for predictor_type-derived outputs.

### use_covariates
- features_only -> R
- features_plus_covariates -> RC
- covariates_only -> C

## Notes
- Mapping is applied only for reporting artifacts (tables/figures), not for core metric computation logic.
- Exact rows are still used as the baseline for Optimality_Gap% and Time_Ratio calculations.

# Experiment Data Analysis - Complete Workflow Documentation

## Overview

This notebook analyzes aggregate Benders decomposition experiment results. It compares Exact Benders runs against ML-augmented Benders variants and creates cleaned datasets, audit outputs, summary tables, and publication-oriented figures.

The two main derived metrics are:

- **Optimality_Gap%**: relative objective-value difference between an ML row and the matching Exact row for the same `instance_filename`.
- **Time_Ratio**: ML solution time divided by the matching Exact solution time for the same `instance_filename`.

The Exact row is the basis for metric eligibility. If an instance has an Exact row with `stopping_reason == "OPTIMAL"`, all rows for that instance can receive metrics, including ML rows whose own `stopping_reason` is not `OPTIMAL`.

---

## Data Processing Pipeline

### Phase 1: Load and clean data

1. Read the aggregate experiment workbook or CSV.
2. Create `deterministic_instance` and `instance_filename_filile_name` from the final path component of `instance_filename`.
3. Retain only rows whose deterministic instance name is listed in `INSTANCES_TO_INCLUDE`.
4. Remove rows where `ml_method_type == "ML_at_integer_exact_at_LP"`.
5. Remove duplicate experiment rows using the configured duplicate identity columns and the two-stage priority rule:
   - First priority: `time_limit_sec == 10800`, then `time_limit_sec == 3600`, then all-null `time_limit_sec` groups.
   - Second priority: prefer rows where `MIPGap_target` is neither missing nor zero.
   - Final selection: when more than one row still qualifies, select one row randomly using the configured random seed.
6. Save the deduplicated data immediately to `deduplicated_experiment_data.csv`.

### Phase 2: Exact-baseline data groups

After cleaning, the notebook adds `determisitic_instance_name` as the final column. This value is the substring after the last `/` in `instance_filename`.

The cleaned data is split into three non-overlapping groups:

- `cleaned_data`: all rows after deterministic-instance filtering and duplicate removal.
- `exact_optimal_baseline`: rows whose `instance_filename` has an Exact row with `stopping_reason == "OPTIMAL"`. ML rows are included by instance, regardless of their own stopping status.
- `exact_nonoptimal_baseline`: the complement of `exact_optimal_baseline` within `cleaned_data`.

`exact_optimal_baseline` plus `exact_nonoptimal_baseline` reconstructs `cleaned_data`.

### Phase 3: Metric computation

For each ML row whose instance has an Exact OPTIMAL baseline:

1. Find the Exact row with the same `instance_filename`.
2. Compute `Optimality_Gap% = (ML_Obj - Exact_Obj) / Exact_Obj`.
3. Compute `Time_Ratio = ML_Time / Exact_Time`.
4. Leave metrics missing if no Exact OPTIMAL baseline exists.

For eligible Exact rows, the notebook sets `Optimality_Gap% = 0` and `Time_Ratio = 1.0`.

### Phase 4: Analysis method filter

After the three cleaned group files are generated, metric tables and plots keep only:

- Exact rows
- `ML_Augmented_BD`
- `Uncertainty_Similarity`

The nonoptimal percentage table is still written with the requested method columns.

---

## Analysis Structure

The analysis is generated across three dimensions.

### Dimension 1: Data group

- `cleaned_data/`
- `exact_optimal_baseline/`
- `exact_nonoptimal_baseline/`

### Dimension 2: Problem class

- `total/`: all problem classes combined.
- `[problem_class]/`: one folder for each distinct problem class in the group.

### Dimension 3: ML callback type

Each problem-class folder contains these callback folders:

- `total/`: all callback types combined.
- `ML_at_integer_exact_at_LP/`
- `ML_at_both/`
- `ML_at_LP_exact_at_integer/`

Each callback folder contains:

- `tables/`: CSV summary tables plus `analysis_input_data.csv` and `method_optimality_summary.xlsx` (sheet 1: overall, sheet 2: by `instance_filename`).
- `figures/line_plots/`: metric trend plots.
- `figures/boxplots/`: metric distribution plots.

The notebook also writes `cleaned_data_analysis.xlsx`, which includes the cleaned dataset after method filtering (`ANALYSIS_ML_METHODS`) and the `num_subproblems` filter `{100, 300, 500}`.

---

## Main Outputs

```text
output_dir/
|-- processed_experiment_analysis.xlsx
|-- deduplicated_experiment_data.csv
|-- cleaned_data.csv
|-- exact_optimal_baseline.csv
|-- exact_nonoptimal_baseline.csv
|-- cleaned_data_analysis.xlsx
|-- cleaned_data/
|   |-- total/
|   |   |-- total/
|   |   |   |-- tables/
|   |   |   `-- figures/
|   |   |       |-- line_plots/
|   |   |       `-- boxplots/
|   |   `-- [callback_type]/
|   `-- [problem_class]/
|-- exact_optimal_baseline/
`-- exact_nonoptimal_baseline/
```

The Excel workbook contains the three processed data-group sheets and audit sheets when available.

# Experiment Data Dictionary, Cleaning Rules, Derived Metrics, and Output Specification

## 1. Purpose

This dataset is used to compare multiple Benders decomposition variants across different instances and problem classes. The methods fall into two broad categories:

- **Exact methods**: the standard Benders decomposition baseline
- **ML methods**: machine-learning-augmented Benders decomposition variants

The analysis focuses mainly on two performance dimensions:

- **Optimality_Gap%**
- **Time_Ratio**

Both of these are defined for each ML run relative to the corresponding Exact run on the same instance.

---

## 2. Core Columns and Their Meaning

### 2.1 Instance and problem descriptors

- **problem_class**  
  The class of optimization problem being solved.

- **instance_filename**  
  Full instance path or identifier.

- **instance_filename_filile_name**  
  A derived compatibility column created from `instance_filename`.

- **determisitic_instance_name**  
  A derived column appended at export time. It stores the substring after the last `/` in `instance_filename`.  
  Example:  
  `folder1/folder2/my_instance.txt` -> `my_instance.txt`

- **num_subproblems**  
  Number of subproblems in the Benders decomposition.

- **num_first_stage_variables**  
  Number of first-stage decision variables.

---

## 3. Method taxonomy and relationships

### 3.1 Broad method category

- **method_category**  
  Indicates whether the run belongs to:
  - `Exact`
  - `ML`

This is the main high-level split between the standard Benders baseline and its ML-based variants.

---

### 3.2 Exact methods

The Exact group consists of only one method: the standard Benders decomposition baseline.

Relevant columns for Exact rows:

- **method_category = `Exact`**
- **ml_method_type = null**
- **ml_callback_type = null**
- **exact_callback_type = `int_and_LP` or null**
- **predictor_type = null**
- **selected_subproblems_ratio = null**

Interpretation:

- Exact runs solve all required subproblems exactly.
- Therefore, no prediction-related settings apply.

---

### 3.3 ML methods

The ML group contains multiple ML-augmented Benders variants.

- **method_category = `ML`**
- **ml_method_type** identifies which ML variant is used.

Possible values of `ml_method_type` include:

- `ML_Augmented_BD`
- `ML_Augmented_BD_with_duals`
- `Uncertainty_Similarity`
- `Random_Sampling`

---

### 3.4 Callback behavior

Two callback-related columns exist because Exact and ML methods use different configuration fields.

#### For ML methods
- **ml_callback_type**
  Possible values:
  - `ML_at_integer_exact_at_LP`
  - `ML_at_both`
  - `ML_at_LP_exact_at_integer`
  - `null` for Exact rows

#### For Exact methods
- **exact_callback_type**
  Possible values:
  - `int_and_LP`
  - `null` for ML rows

Interpretation:

These columns describe whether cuts are generated only at integer solutions or also at LP relaxations.

---

### 3.5 Predictor specification

- **predictor_type**

This is only relevant for certain ML methods and is null for Exact rows.

For the following ML methods:
- `ML_Augmented_BD`
- `ML_Augmented_BD_with_duals`
- `Uncertainty_Similarity`

possible values include:
- `TREE`
- `CLUSTERING`

For:
- `Random_Sampling`

the value is:
- `null`

because that method does not have prediction-function variants in this sense.

---

### 3.6 Exactly solved subproblem ratio

- **selected_subproblems_ratio**

This column is relevant for ML methods and null for Exact rows.

Interpretation:

It gives the fraction of subproblems solved exactly.  
Example:
- `0.15` means 15% of subproblems are solved exactly
- the remaining 85% use predicted cuts

Thus, this column directly controls the amount of exact information injected into the ML method.

---

### 3.7 Covariate usage

- **use_covariates**

Possible values:
- `features_only`
- `features_plus_covariates`
- `covariates_only`
- `null` for rows where this setting is not applicable

This field is especially relevant for `ML_Augmented_BD`.

---

### 3.8 Termination-related columns

- **stopping_reason**
  - `OPTIMAL`
  - `TIME_LIMIT`

- **MIPGap_Stop**
  If `TIME_LIMIT` is hit, this stores the gap at termination.

---

## 4. Metrics to Construct

Two derived performance metrics are required.

### 4.1 Matching rule

For each ML row, find the corresponding Exact row using:

- the same **instance_filename**

The Exact row is the baseline for that instance. Metric eligibility is based on the Exact row status, not the ML row status.

---

### 4.2 Optimality_Gap%

For each ML row:

$$
\text{Optimality\_Gap\%} =
\frac{
\text{original\_Prob\_Obj\_Value}_{ML} -
\text{original\_Prob\_Obj\_Value}_{Exact}
}{
\text{original\_Prob\_Obj\_Value}_{Exact}
}
$$

Notes:

- This is a relative gap with respect to the Exact method.
- The output column name is `Optimality_Gap%`.
- Depending on reporting preference, this may remain as a decimal ratio or later be formatted as a percentage in tables/figures.

For eligible Exact OPTIMAL baseline rows:
- set `Optimality_Gap% = 0`

---

### 4.3 Time_Ratio

For each ML row:

$$
\text{Time\_Ratio} =
\frac{
\text{solution\_time\_sec}_{ML}
}{
\text{solution\_time\_sec}_{Exact}
}
$$

Interpretation:

- Values smaller than 1 indicate an ML method faster than Exact.
- Values larger than 1 indicate slower performance than Exact.

For eligible Exact OPTIMAL baseline rows:
- set `Time_Ratio = 1`

---

## 5. Duplicate removal rule

Before constructing metrics or writing analysis outputs, duplicate experiment rows are removed. The intent is to treat rows as duplicates only when they describe the same instance, method, callback policy, prediction policy, sampling/trial settings, and embedding/training settings.

Two or more rows are considered duplicate candidates when they match exactly on all available columns from this configured identity list:

- `instance_filename`
- `method_category`
- `ml_method_type`
- `ml_callback_type`
- `exact_callback_type`
- `predictor_type`
- `use_prediction`
- `prediction_method`
- `selected_subproblems_ratio`
- `n_trials`
- `m_trials`
- `n_walks`
- `walk_length`
- `feature_dim`
- `window`
- `min_count`
- `sg`

`time_limit_sec` is intentionally excluded from duplicate matching because it is used as the first priority rule after duplicate groups have been identified. `MIPGap_target` is also excluded from matching because it is used as the second priority rule.

### Duplicate-handling rule

For each duplicate candidate group:

1. Apply the `time_limit_sec` priority.
   - If one or more rows have `time_limit_sec == 10800`, keep only those rows for the next stage.
   - Otherwise, if one or more rows have `time_limit_sec == 3600`, keep only those rows for the next stage.
   - Otherwise, if every row has missing `time_limit_sec`, keep all rows for the next stage.
   - Otherwise, keep all rows in the group for the next stage because no preferred time-limit tier is present.
2. Apply the `MIPGap_target` priority to the rows that remain from stage 1.
   - If one or more rows have `MIPGap_target` values that are neither missing nor zero, keep one of those rows.
   - If no remaining row has a nonzero, nonmissing `MIPGap_target`, keep one row from the remaining group.

When more than one row remains acceptable after both priority stages, the implementation selects one row randomly using `DUPLICATE_RANDOM_STATE`. This preserves the requested random choice while making notebook reruns reproducible.

The rows removed by this rule are returned separately as `removed_duplicates` and are exported in the Excel workbook when nonempty. The deduplicated dataset is also written to a standalone CSV file immediately after duplicate removal.

This deduplication happens before:
- filtering instances by the Exact baseline stopping status
- matching ML rows to Exact rows
- computing `Optimality_Gap%`
- computing `Time_Ratio`
- generating summary tables and plots

---

## 6. Short names used in analysis

For reporting and figure legends, the following short names are used:

- `ML_Augmented_BD_with_duals` → `ml_dl`
- `ML_Augmented_BD` → `ml_ftr`
- `Uncertainty_Similarity` → `ml_uncrtn`
- `Random_Sampling` → `ml_smplng`
- `Exact` → `exact`

These short names are used in tables, plots, and derived analysis sheets.

---

## 7. Output Excel file specification

The output should be an Excel workbook that contains:

### 7.1 Main processed dataset sheet
A sheet containing the cleaned and enriched dataset with:
- duplicates removed using the two-stage `time_limit_sec` and `MIPGap_target` priority rule
- `instance_filename_filile_name`
- short-name columns
- `Optimality_Gap%`
- `Time_Ratio`

### 7.2 Analysis sheets
Additional sheets should contain summary tables such as:

- summary by ML method over the two metrics
- summary by `selected_subproblems_ratio`
- sensitivity by `ml_method_type` and `prediction_method`
- sensitivity by `use_covariates`
- sensitivity by `n_trials`
- sensitivity by `m_trials`

Only rows for which the relevant columns are meaningful should be included in each sensitivity table.

---

## 8. Folder structure for outputs

The code should write all outputs into a user-specified output directory.

Inside the output directory, create:

- `figures/`  
  stores all generated figures

- `tables/`  
  stores exported table files such as `.csv`

- the standalone deduplicated CSV file  
  stored directly in the output directory

- the final Excel workbook  
  stored directly in the output directory

Recommended structure:

```text
output_dir/
├── processed_experiment_analysis.xlsx
├── deduplicated_experiment_data.csv
├── figures/
│   ├── selected_ratio_vs_optimality_gap.png
│   ├── selected_ratio_vs_time_ratio.png
│   └── ...
└── tables/
    ├── summary_by_method.csv
    ├── summary_by_method_prediction_method.csv
    ├── summary_by_use_covariates.csv
    ├── summary_by_n_trials.csv
    └── summary_by_m_trials.csv
```


## 9. Post-Dedup Exact-Baseline Data Groups

After duplicate removal, the workflow uses only rows where `method_category == "Exact"` to split the cleaned data by instance.

Rules:

- If an instance has an Exact row with `stopping_reason == "OPTIMAL"`, all rows for that `instance_filename` go to `exact_optimal_baseline`.
- ML rows follow their corresponding Exact instance, regardless of the ML row's own `stopping_reason`.
- All remaining cleaned rows go to `exact_nonoptimal_baseline`.
- `exact_optimal_baseline` and `exact_nonoptimal_baseline` do not overlap, and together they reconstruct `cleaned_data`.

Why this is needed: `Optimality_Gap%` and `Time_Ratio` require a valid Exact reference for each instance. Using the Exact row as the basis keeps non-OPTIMAL ML rows available for metric calculation when the Exact baseline is valid.


---

In all line plots displaying the `Time_Ratio` metric, a horizontal dashed reference line is drawn at `y=1`. This line marks the equivalence point where ML method runtime equals Exact method runtime. Values below 1 indicate ML speedup; values above 1 indicate ML slowdown.

## 10. Additional Analysis Filters and Stopping-Reason Based Grouping

### 10.3 Time_Ratio Reference Line

### 10.1 Number of Subproblems Filter

Note: Both groups follow the same `num_subproblems` filter, and analysis (metrics, tables, figures) is generated for each group separately.

For analysis purposes, only instances with `num_subproblems` in the set `{100, 300, 500}` are included. This filter is applied to all analysis data groups after metrics are computed, ensuring consistent problem size representation across all comparisons.

- All other rows (Exact rows without OPTIMAL status, ML rows without matching Exact-OPTIMAL or their own non-OPTIMAL status) go to `NOT_all_Optimal`.

### 10.2 All_Optimal vs NOT_all_Optimal Grouping- If an ML row also has `stopping_reason == "OPTIMAL"`, place both the Exact and ML rows in the `All_Optimal` group.

- For each such Exact-OPTIMAL row, find all corresponding ML rows (same `instance_filename`, `ml_method_type` in `ANALYSIS_ML_METHODS`).

In addition to the Exact-baseline-status split (`exact_optimal_baseline` and `exact_nonoptimal_baseline`), two additional data groups are created based on stopping-reason alignment between Exact and ML methods:- For each `instance_filename`, identify all Exact rows with `stopping_reason == "OPTIMAL"` (should typically be one Exact row per instance after deduplication).

Grouping logic:

**All_Optimal Group:** Contains row pairs (or individual Exact rows) where both the Exact method AND the corresponding ML method (from `ANALYSIS_ML_METHODS`: `ML_Augmented_BD`, `Random_Sampling`) achieved `stopping_reason == "OPTIMAL"`. These represent instances where ML successfully matched the Exact solver's solution quality.

**NOT_all_Optimal Group:** Contains all remaining rows—those where at least one of the two methods (Exact or ML) did not reach `stopping_reason == "OPTIMAL"`. These represent cases where either the Exact method did not solve optimally, or the ML method failed to match the Exact solution quality, or both.

## 9. Post-Dedup Exact-Baseline Data Groups

After duplicate removal, the workflow uses only rows where `method_category == "Exact"` to split the cleaned data by instance.

Rules:

- If an instance has an Exact row with `stopping_reason == "OPTIMAL"`, all rows for that `instance_filename` go to `exact_optimal_baseline`.
- ML rows follow their corresponding Exact instance, regardless of the ML row's own `stopping_reason`.
- All remaining cleaned rows go to `exact_nonoptimal_baseline`.
- `exact_optimal_baseline` and `exact_nonoptimal_baseline` do not overlap, and together they reconstruct `cleaned_data`.

Why this is needed: `Optimality_Gap%` and `Time_Ratio` require a valid Exact reference for each instance. Using the Exact row as the basis keeps non-OPTIMAL ML rows available for metric calculation when the Exact baseline is valid.

---

## 10. Additional Analysis Filters and Stopping-Reason Based Grouping

### 10.1 Number of Subproblems Filter

For analysis purposes, only instances with `num_subproblems` in the set `{100, 300, 500}` are included. This filter is applied to all analysis data groups after metrics are computed, ensuring consistent problem size representation across all comparisons.

### 10.2 All_Optimal vs NOT_all_Optimal Grouping

In addition to the Exact-baseline-status split (`exact_optimal_baseline` and `exact_nonoptimal_baseline`), two additional data groups are created based on stopping-reason alignment between Exact and ML methods:

**All_Optimal Group:** Contains row pairs (or individual Exact rows) where both the Exact method AND the corresponding ML method (from `ANALYSIS_ML_METHODS`: `ML_Augmented_BD`, `Random_Sampling`) achieved `stopping_reason == "OPTIMAL"`. These represent instances where ML successfully matched the Exact solver's solution quality.

**NOT_all_Optimal Group:** Contains all remaining rows—those where at least one of the two methods (Exact or ML) did not reach `stopping_reason == "OPTIMAL"`. These represent cases where either the Exact method did not solve optimally, or the ML method failed to match the Exact solution quality, or both.

Grouping logic:
- For each `instance_filename`, identify all Exact rows with `stopping_reason == "OPTIMAL"` (should typically be one Exact row per instance after deduplication).
- For each such Exact-OPTIMAL row, find all corresponding ML rows (same `instance_filename`, `ml_method_type` in `ANALYSIS_ML_METHODS`).
- If an ML row also has `stopping_reason == "OPTIMAL"`, place both the Exact and ML rows in the `All_Optimal` group.
- All other rows (Exact rows without OPTIMAL status, ML rows without matching Exact-OPTIMAL or their own non-OPTIMAL status) go to `NOT_all_Optimal`.

Note: Both groups follow the same `num_subproblems` filter, and analysis (metrics, tables, figures) is generated for each group separately.

### 10.3 Time_Ratio Reference Line

In all line plots displaying the `Time_Ratio` metric, a horizontal dashed reference line is drawn at `y=1`. This line marks the equivalence point where ML method runtime equals Exact method runtime. Values below 1 indicate ML speedup; values above 1 indicate ML slowdown.

In [47]:
import os
from pathlib import Path
import warnings
import gc

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Match the Matplotlib-only plot style used in experiments.ipynb.
plt.style.use("default")
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


## User Inputs

Set the source workbook and output filenames used by the rest of the workflow.


In [48]:
# =========================
# USER INPUTS
# =========================

INPUT_FILE = r"/mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Aggregate_of_experiments.xlsx"
OUTPUT_DIR = r"/mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output"

OUTPUT_EXCEL_NAME = "processed_experiment_analysis.xlsx"
OUTPUT_DEDUPED_CSV_NAME = "deduplicated_experiment_data.csv"
OUTPUT_EXPERIMENT_COVERAGE_EXCEL_NAME = "experiment_coverage_counts.xlsx"

## Analysis Configuration

Define duplicate identity columns, callback folders, naming maps, and metric display helpers.


## Duplicate and Callback Configuration


In [49]:
# =========================
# CONFIGURATION
# =========================

# Rows are considered IDENTICAL (duplicate candidates) if and only if they match
# on ALL of these columns. Null/empty values are treated as matching pairs
# (i.e., if both rows have null/NaN in a column, they match on that column).
#
# Note: time_limit_sec and MIPGap_target are NOT part of the duplicate identity.
# Instead, they are used as PRIORITY FIELDS to select one representative from
# each duplicate group using the selection rules below.
#
# SELECTION RULES (applied in order):
# 1. Within each duplicate group, prefer rows by time_limit_sec tier:
#    - First tier: time_limit_sec == 10800
#    - Second tier: time_limit_sec == 3600
#    - Third tier: time_limit_sec is null/NaN
#    If no rows exist in a tier, proceed to the next tier.
#
# 2. From remaining rows after time_limit_sec filtering, prefer rows where
#    MIPGap_target is NOT null and NOT 0.
#
# 3. If multiple acceptable rows remain, randomly select one row using
#    DUPLICATE_RANDOM_STATE for reproducibility.

DUPLICATE_SUBSET = [
    "problem_class",
    "deterministic_instance",
    "num_subproblems",
    "method_category",
    "ml_method_type",
    "ml_callback_type",
    "exact_callback_type",
    "predictor_type",
    "use_prediction",
    "prediction_method",
    "selected_subproblems_ratio",
    "n_trials",
    "m_trials",
    "n_walks",
    "walk_length",
    "feature_dim",
    "window",
    "min_count",
    "sg",
    "use_covariates",
]

CALLBACK_TYPES_TO_ANALYZE = [
    "total",
    "ML_at_integer_exact_at_LP",
    "ML_at_both",
    "ML_at_LP_exact_at_integer",
    "ML_at_both_and_ML_at_LP_exact_at_integer",
]

COMBINED_CALLBACK_TYPE_MAP = {
    "ML_at_both_and_ML_at_LP_exact_at_integer": [
        "ML_at_both",
        "ML_at_LP_exact_at_integer",
    ],
}

# Used only when duplicate-priority rules leave multiple acceptable rows.
DUPLICATE_RANDOM_STATE = 42


SHORT_NAME_MAP = {
    "ML_Augmented_BD_with_duals": "ml_dl",
    "ML_Augmented_BD": "ml_ftr",
    "Uncertainty_Similarity": "ml_uncrtn",
    "Random_Sampling": "ml_smplng",
    "Exact": "exact",
}

METRICS = ["Optimality_Gap%", "Time_Ratio"]


DATA_GROUPS_TO_ANALYZE = [
    ("cleaned_data", "Cleaned data"),
    ("exact_optimal_baseline", "Exact OPTIMAL baseline"),
    ("exact_nonoptimal_baseline", "Exact non-OPTIMAL baseline"),
]

ANALYSIS_ML_METHODS = [
    "ML_Augmented_BD",
    "Random_Sampling",
]


## Method and Data-Group Configuration


In [50]:
NONOPTIMAL_PERCENT_METHOD_COLUMNS = [
    "Exact",
    "ML_Augmented_BD",
    "ML_Augmented_BD_with_duals",
    "Uncertainty_Similarity",
    "Random_Sampling",
]


## Presentation Naming Configuration


In [51]:
PRESENTATION_NAME_MAP = {
    "ml_method_type": {
        "ML_Augmented_BD": "LABD",
        "ML_Augmented_BD_with_duals": "LABD-D",
        "Uncertainty_Similarity": "uncrtn",
        "Random_Sampling": "RND",
    },
    "prediction_method": {
        "KNN": "kNN",
        "TREE": "DT",
        "CLUSTERING": "CLU",
    },
    "predictor_type": {
        "KNN": "kNN",
        "TREE": "DT",
        "CLUSTERING": "CLU",
    },
    "use_covariates": {
        "features_only": "R",
        "features_plus_covariates": "RC",
        "covariates_only": "C",
    },
}


## Presentation Naming Helpers


In [52]:
def metric_display_name(metric):
    if metric == "Optimality_Gap%":
        return "Optimality Gap"
    if metric == "Time_Ratio":
        return "Time Ratio"
    return metric


def _is_blank_label(value):
    if pd.isna(value):
        return True
    text = str(value).strip()
    return text == "" or text.lower() in {"null", "none", "nan", "na"}


def combine_display_names(*parts):
    labels = [str(part).strip() for part in parts if not _is_blank_label(part)]
    if not labels:
        return ""
    return " + ".join(labels)


def apply_presentation_naming(df):
    """Apply publication naming conventions to reporting columns if they exist."""
    out = df.copy()
    for col, mapping in PRESENTATION_NAME_MAP.items():
        if col in out.columns:
            out[col] = out[col].map(lambda x: mapping.get(x, x))
    return out


## Instance Selection

List the deterministic instance filenames retained for this analysis run.


## Selected Subproblem Ratio: Shared Summary Helpers


In [53]:
# =========================
# SELECTED_SUBPROBLEMS_RATIO ANALYSIS HELPERS
# =========================
# These mirror the num_subproblems summary/line-plot outputs, but use
# selected_subproblems_ratio as the table column split and plot x-axis.


def axis_display_name(axis_column):
    if axis_column == "num_subproblems":
        return "Number of Subproblems"
    if axis_column == "selected_subproblems_ratio":
        return "Exact Subproblems Ratio (%)"
    return axis_column.replace("_", " ").title()


def _format_ratio_percent(value):
    if pd.isna(value):
        return ""
    percent = value * 100
    if abs(percent - round(percent)) < 1e-6:
        return str(int(round(percent)))
    return str(round(percent, 2)).replace(".", "p")


def axis_column_prefix(axis_column, value):
    if axis_column == "num_subproblems":
        return f"sp{value}"
    if axis_column == "selected_subproblems_ratio":
        return f"ratio_{_format_ratio_percent(value)}"
    return f"{axis_column}_{str(value).replace('.', 'p')}"


def build_summary_by_axis(df, row_columns, output_column, axis_column="num_subproblems", metrics=METRICS):
    """Build a summary table over a configurable analysis axis."""
    df_ml = apply_presentation_naming(df[df["method_category"] == "ML"].copy())

    if len(df_ml) == 0:
        return pd.DataFrame()

    results = []
    combinations = df_ml[row_columns].drop_duplicates()

    for _, combo_row in combinations.iterrows():
        values = [combo_row[col] for col in row_columns]
        if pd.isna(values[0]):
            continue

        subset = df_ml.copy()
        for col, value in zip(row_columns, values):
            if pd.isna(value):
                subset = subset[subset[col].isna()]
            else:
                subset = subset[subset[col] == value]

        if len(subset) == 0:
            continue

        row_label = values[0] if len(row_columns) == 1 else combine_display_names(*values)
        row_dict = {output_column: row_label}
        axis_values = sorted([x for x in subset[axis_column].unique() if pd.notna(x)])

        for axis_value in axis_values:
            axis_subset = subset[subset[axis_column] == axis_value]
            axis_prefix = axis_column_prefix(axis_column, axis_value)

            for metric in metrics:
                metric_values = axis_subset[metric].dropna()
                if len(metric_values) > 0:
                    row_dict[f"{axis_prefix}_{metric}_min"] = metric_values.min()
                    row_dict[f"{axis_prefix}_{metric}_max"] = metric_values.max()
                    row_dict[f"{axis_prefix}_{metric}_mean"] = metric_values.mean()
                    row_dict[f"{axis_prefix}_{metric}_median"] = metric_values.median()

        for metric in metrics:
            metric_values = subset[metric].dropna()
            if len(metric_values) > 0:
                row_dict[f"total_{metric}_min"] = metric_values.min()
                row_dict[f"total_{metric}_max"] = metric_values.max()
                row_dict[f"total_{metric}_mean"] = metric_values.mean()
                row_dict[f"total_{metric}_median"] = metric_values.median()

        results.append(row_dict)

    return pd.DataFrame(results)


## Selected Subproblem Ratio: Summary Wrappers


In [54]:
def build_summary_by_selected_subproblems_ratio_ml_method(df, metrics=METRICS):
    return build_summary_by_axis(
        df,
        row_columns=["ml_method_type"],
        output_column="ml_method_type",
        axis_column="selected_subproblems_ratio",
        metrics=metrics,
    )


def build_summary_by_selected_subproblems_ratio_ml_method_and_predictor(df, metrics=METRICS):
    return build_summary_by_axis(
        df,
        row_columns=["ml_method_type", "predictor_type"],
        output_column="method_predictor",
        axis_column="selected_subproblems_ratio",
        metrics=metrics,
    )


def build_summary_by_selected_subproblems_ratio_ml_method_and_covariates(df, metrics=METRICS):
    return build_summary_by_axis(
        df,
        row_columns=["ml_method_type", "use_covariates"],
        output_column="method_covariates",
        axis_column="selected_subproblems_ratio",
        metrics=metrics,
    )


def build_summary_by_selected_subproblems_ratio_ml_method_predictor_covariates(df, metrics=METRICS):
    return build_summary_by_axis(
        df,
        row_columns=["ml_method_type", "predictor_type", "use_covariates"],
        output_column="method_predictor_covariates",
        axis_column="selected_subproblems_ratio",
        metrics=metrics,
    )


## Selected Subproblem Ratio: Shared Plot Helper


In [55]:
def _ensure_time_ratio_tick(ax):
    ticks = list(ax.get_yticks())
    if not any(abs(tick - 1.0) < 1e-9 for tick in ticks):
        ticks.append(1.0)
        ax.set_yticks(sorted(ticks))


def _plot_time_ratio_layouts(grouped, axis_column, title_base, output_path, legend_fontsize):
    axis_label = axis_display_name(axis_column)

    def _x_values(data):
        x_values = data[axis_column]
        if axis_column == "selected_subproblems_ratio":
            x_values = x_values * 100
        return x_values

    # Layout 1: Full-scale figure
    plt.figure(figsize=(12, 6))
    for combo in grouped["combination"].unique():
        combo_data = grouped[grouped["combination"] == combo].sort_values(axis_column)
        plt.plot(
            _x_values(combo_data),
            combo_data["Time_Ratio"],
            marker="o",
            label=combo,
        )
    plt.xlabel(axis_label)
    plt.ylabel("Time_Ratio")
    plt.title(title_base)
    if legend_fontsize:
        plt.legend(fontsize=legend_fontsize)
    else:
        plt.legend()
    plt.axhline(y=1, color="gray", linestyle="--", linewidth=1.5, alpha=0.7)
    _ensure_time_ratio_tick(plt.gca())
    plt.tight_layout()
    if output_path:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved: {output_path}")
    plt.close()

    # Layout 1: Zoomed figure
    zoomed_path = None
    if output_path:
        zoomed_path = output_path.with_name(f"{output_path.stem}_zoom_0_3{output_path.suffix}")
    plt.figure(figsize=(12, 6))
    for combo in grouped["combination"].unique():
        combo_data = grouped[grouped["combination"] == combo].sort_values(axis_column)
        plt.plot(
            _x_values(combo_data),
            combo_data["Time_Ratio"],
            marker="o",
            label=combo,
        )
    plt.xlabel(axis_label)
    plt.ylabel("Time_Ratio")
    plt.title(f"{title_base} - Zoomed View (Y-axis: 0 to 3)")
    plt.ylim(0, 3)
    if legend_fontsize:
        plt.legend(fontsize=legend_fontsize)
    else:
        plt.legend()
    plt.axhline(y=1, color="gray", linestyle="--", linewidth=1.5, alpha=0.7)
    _ensure_time_ratio_tick(plt.gca())
    plt.tight_layout()
    if zoomed_path:
        zoomed_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(zoomed_path, dpi=300, bbox_inches="tight")
        print(f"Saved: {zoomed_path}")
    plt.close()

    # Layout 2: Side-by-side figure
    side_by_side_path = None
    if output_path:
        side_by_side_path = output_path.with_name(
            f"{output_path.stem}_side_by_side{output_path.suffix}"
        )
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)
    for combo in grouped["combination"].unique():
        combo_data = grouped[grouped["combination"] == combo].sort_values(axis_column)
        axes[0].plot(
            _x_values(combo_data),
            combo_data["Time_Ratio"],
            marker="o",
            label=combo,
        )
        axes[1].plot(
            _x_values(combo_data),
            combo_data["Time_Ratio"],
            marker="o",
            label=combo,
        )
    axes[0].set_xlabel(axis_label)
    axes[0].set_ylabel("Time_Ratio")
    axes[0].set_title("Full Scale Performance")
    axes[0].axhline(y=1, color="gray", linestyle="--", linewidth=1.5, alpha=0.7)
    _ensure_time_ratio_tick(axes[0])
    axes[1].set_xlabel(axis_label)
    axes[1].set_ylabel("Time_Ratio")
    axes[1].set_title("Focused Range (0 to 3)")
    axes[1].set_ylim(0, 3)
    axes[1].axhline(y=1, color="gray", linestyle="--", linewidth=1.5, alpha=0.7)
    _ensure_time_ratio_tick(axes[1])
    if legend_fontsize:
        axes[0].legend(fontsize=legend_fontsize)
    else:
        axes[0].legend()
    fig.tight_layout()
    if side_by_side_path:
        side_by_side_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(side_by_side_path, dpi=300, bbox_inches="tight")
        print(f"Saved: {side_by_side_path}")
    plt.close(fig)


def plot_metric_vs_axis(df, metric, output_path, filter_dict, axis_column, line_columns, title_suffix):
    try:
        df_plot = apply_presentation_naming(df[df["method_category"] == "ML"].copy())

        if filter_dict:
            for col, val in filter_dict.items():
                if val is not None and val != "total":
                    df_plot = df_plot[df_plot[col] == val]

        if len(line_columns) == 1:
            df_plot["combination"] = df_plot[line_columns[0]]
        else:
            df_plot["combination"] = df_plot.apply(
                lambda row: combine_display_names(*[row[col] for col in line_columns]),
                axis=1,
            )

        grouped = df_plot.groupby([axis_column, "combination"])[metric].mean().reset_index()

        if len(grouped) == 0:
            print(f"No data for metric {metric}")
            return None

        axis_label = axis_display_name(axis_column)
        title_base = f"{metric_display_name(metric)} vs {axis_label} by {title_suffix}"
        legend_fontsize = 8 if len(line_columns) > 1 else None

        if metric == "Time_Ratio":
            _plot_time_ratio_layouts(
                grouped,
                axis_column=axis_column,
                title_base=title_base,
                output_path=output_path,
                legend_fontsize=legend_fontsize,
            )
            return None

        plt.figure(figsize=(12 if len(line_columns) > 1 else 10, 6))

        for combo in grouped["combination"].unique():
            combo_data = grouped[grouped["combination"] == combo].sort_values(axis_column)
            x_values = combo_data[axis_column]
            if axis_column == "selected_subproblems_ratio":
                x_values = x_values * 100
            plt.plot(
                x_values,
                combo_data[metric],
                marker="o",
                label=combo,
            )

        plt.xlabel(axis_label)
        plt.ylabel(metric)
        plt.title(title_base)
        if legend_fontsize:
            plt.legend(fontsize=legend_fontsize)
        else:
            plt.legend()

        plt.tight_layout()

        if output_path:
            output_path.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(output_path, dpi=300, bbox_inches="tight")
            print(f"Saved: {output_path}")

        plt.close()
        return None
    except Exception as e:
        print(f"Error plotting metric {metric}: {e}")
        plt.close()
        return None

## Selected Subproblem Ratio: Plot Wrappers


In [56]:
def plot_metric_vs_selected_subproblems_ratio_by_ml_method(df, metric="Optimality_Gap%", output_path=None, filter_dict=None):
    return plot_metric_vs_axis(
        df,
        metric=metric,
        output_path=output_path,
        filter_dict=filter_dict,
        axis_column="selected_subproblems_ratio",
        line_columns=["ml_method_type"],
        title_suffix="ML Method",
    )


def plot_metric_vs_selected_subproblems_ratio_by_predictor(df, metric="Optimality_Gap%", output_path=None, filter_dict=None):
    return plot_metric_vs_axis(
        df,
        metric=metric,
        output_path=output_path,
        filter_dict=filter_dict,
        axis_column="selected_subproblems_ratio",
        line_columns=["ml_method_type", "predictor_type"],
        title_suffix="ML Method and Predictor",
    )


def plot_metric_vs_selected_subproblems_ratio_by_covariates(df, metric="Optimality_Gap%", output_path=None, filter_dict=None):
    return plot_metric_vs_axis(
        df,
        metric=metric,
        output_path=output_path,
        filter_dict=filter_dict,
        axis_column="selected_subproblems_ratio",
        line_columns=["ml_method_type", "use_covariates"],
        title_suffix="ML Method and Covariates",
    )


def plot_metric_vs_selected_subproblems_ratio_by_predictor_covariates(df, metric="Optimality_Gap%", output_path=None, filter_dict=None):
    return plot_metric_vs_axis(
        df,
        metric=metric,
        output_path=output_path,
        filter_dict=filter_dict,
        axis_column="selected_subproblems_ratio",
        line_columns=["ml_method_type", "predictor_type", "use_covariates"],
        title_suffix="ML Method, Predictor, and Covariates",
    )


## Summary by Selected Subproblem Ratio

Build table summaries that compare ML methods across selected-subproblem ratios.


In [57]:

# =========================
# DETERMINISTIC INSTANCE FILTERING
# =========================

# Specify which stochastic instance files to include in the analysis
# Only rows with deterministic_instance matching these patterns will be retained
INSTANCES_TO_INCLUDE = [
    "p4_stochastic.txt",
    "p5_stochastic.txt",
    "p10_stochastic.txt",
    "p20_stochastic.txt",
    "p50_stochastic.txt",
    "p51_stochastic.txt",
    "r01.1.stoch.json",
    "r01.2.stoch.json",
    "r02.1.stoch.json",
    "r02.2.stoch.json",
    "r04.1.stoch.json",
    "r04.2.stoch.json",
    "r05.2.stoch.json"
]

# Add more patterns as needed
# INSTANCES_TO_INCLUDE = [
#     "p10_stochastic.txt",
#     "p5_stochastic.txt",
#     "r01.1.stoch.json",
#     "*.stoch.*",  # Add wildcards if needed
# ]


## Plotting Helpers for Selected Subproblem Ratio

Create line plots that track metrics as selected-subproblem ratios change.


In [58]:
# Summary and analysis functions will be defined below
# (All functions defined within the notebook for self-contained execution)

In [59]:
# Summary and analysis functions defined in this notebook for self-contained execution

def build_summary_by_ml_method(df, metrics=METRICS):
    df_ml = df[df["method_category"] == "ML"].copy()

    if len(df_ml) == 0:
        return pd.DataFrame()

    results = []

    for method in df_ml["ml_method_type"].unique():
        if pd.isna(method):
            continue

        method_data = df_ml[df_ml["ml_method_type"] == method]
        row_dict = {"ml_method_type": method}

        subproblem_counts = sorted([x for x in method_data["num_subproblems"].unique() if pd.notna(x)])

        for count in subproblem_counts:
            subset = method_data[method_data["num_subproblems"] == count]

            for metric in metrics:
                metric_values = subset[metric].dropna()

                if len(metric_values) > 0:
                    row_dict[f"sp{count}_{metric}_min"] = metric_values.min()
                    row_dict[f"sp{count}_{metric}_max"] = metric_values.max()
                    row_dict[f"sp{count}_{metric}_mean"] = metric_values.mean()
                    row_dict[f"sp{count}_{metric}_median"] = metric_values.median()

        for metric in metrics:
            metric_values = method_data[metric].dropna()
            if len(metric_values) > 0:
                row_dict[f"total_{metric}_min"] = metric_values.min()
                row_dict[f"total_{metric}_max"] = metric_values.max()
                row_dict[f"total_{metric}_mean"] = metric_values.mean()
                row_dict[f"total_{metric}_median"] = metric_values.median()

        results.append(row_dict)

    return pd.DataFrame(results)


def build_summary_by_ml_method_and_predictor(df, metrics=METRICS):
    df_ml = df[df["method_category"] == "ML"].copy()

    if len(df_ml) == 0:
        return pd.DataFrame()

    results = []

    combinations = df_ml[["ml_method_type", "predictor_type"]].drop_duplicates()

    for _, row in combinations.iterrows():
        method = row["ml_method_type"]
        predictor = row["predictor_type"]

        if pd.isna(method):
            continue

        subset = df_ml[df_ml["ml_method_type"] == method]
        if pd.isna(predictor):
            subset = subset[subset["predictor_type"].isna()]
        else:
            subset = subset[subset["predictor_type"] == predictor]

        if len(subset) == 0:
            continue

        combined_name = combine_display_names(method, predictor)
        row_dict = {"method_predictor": combined_name}

        subproblem_counts = sorted([x for x in subset["num_subproblems"].unique() if pd.notna(x)])

        for count in subproblem_counts:
            sp_subset = subset[subset["num_subproblems"] == count]

            for metric in metrics:
                metric_values = sp_subset[metric].dropna()

                if len(metric_values) > 0:
                    row_dict[f"sp{count}_{metric}_min"] = metric_values.min()
                    row_dict[f"sp{count}_{metric}_max"] = metric_values.max()
                    row_dict[f"sp{count}_{metric}_mean"] = metric_values.mean()
                    row_dict[f"sp{count}_{metric}_median"] = metric_values.median()

        for metric in metrics:
            metric_values = subset[metric].dropna()
            if len(metric_values) > 0:
                row_dict[f"total_{metric}_min"] = metric_values.min()
                row_dict[f"total_{metric}_max"] = metric_values.max()
                row_dict[f"total_{metric}_mean"] = metric_values.mean()
                row_dict[f"total_{metric}_median"] = metric_values.median()

        results.append(row_dict)

    return pd.DataFrame(results)


def build_summary_by_ml_method_and_covariates(df, metrics=METRICS):
    df_ml = df[df["method_category"] == "ML"].copy()

    if len(df_ml) == 0:
        return pd.DataFrame()

    results = []

    combinations = df_ml[["ml_method_type", "use_covariates"]].drop_duplicates()

    for _, row in combinations.iterrows():
        method = row["ml_method_type"]
        covariates = row["use_covariates"]

        if pd.isna(method):
            continue

        subset = df_ml[df_ml["ml_method_type"] == method]
        if pd.isna(covariates):
            subset = subset[subset["use_covariates"].isna()]
        else:
            subset = subset[subset["use_covariates"] == covariates]

        if len(subset) == 0:
            continue

        combined_name = combine_display_names(method, covariates)
        row_dict = {"method_covariates": combined_name}

        subproblem_counts = sorted([x for x in subset["num_subproblems"].unique() if pd.notna(x)])

        for count in subproblem_counts:
            sp_subset = subset[subset["num_subproblems"] == count]

            for metric in metrics:
                metric_values = sp_subset[metric].dropna()

                if len(metric_values) > 0:
                    row_dict[f"sp{count}_{metric}_min"] = metric_values.min()
                    row_dict[f"sp{count}_{metric}_max"] = metric_values.max()
                    row_dict[f"sp{count}_{metric}_mean"] = metric_values.mean()
                    row_dict[f"sp{count}_{metric}_median"] = metric_values.median()

        for metric in metrics:
            metric_values = subset[metric].dropna()
            if len(metric_values) > 0:
                row_dict[f"total_{metric}_min"] = metric_values.min()
                row_dict[f"total_{metric}_max"] = metric_values.max()
                row_dict[f"total_{metric}_mean"] = metric_values.mean()
                row_dict[f"total_{metric}_median"] = metric_values.median()

        results.append(row_dict)

    return pd.DataFrame(results)


def build_summary_by_ml_method_predictor_covariates(df, metrics=METRICS):
    df_ml = df[df["method_category"] == "ML"].copy()

    if len(df_ml) == 0:
        return pd.DataFrame()

    results = []

    combinations = df_ml[["ml_method_type", "predictor_type", "use_covariates"]].drop_duplicates()

    for _, row in combinations.iterrows():
        method = row["ml_method_type"]
        predictor = row["predictor_type"]
        covariates = row["use_covariates"]

        if pd.isna(method):
            continue

        subset = df_ml[df_ml["ml_method_type"] == method]
        if pd.isna(predictor):
            subset = subset[subset["predictor_type"].isna()]
        else:
            subset = subset[subset["predictor_type"] == predictor]

        if pd.isna(covariates):
            subset = subset[subset["use_covariates"].isna()]
        else:
            subset = subset[subset["use_covariates"] == covariates]

        if len(subset) == 0:
            continue

        combined_name = combine_display_names(method, predictor, covariates)
        row_dict = {"method_predictor_covariates": combined_name}

        subproblem_counts = sorted([x for x in subset["num_subproblems"].unique() if pd.notna(x)])

        for count in subproblem_counts:
            sp_subset = subset[subset["num_subproblems"] == count]

            for metric in metrics:
                metric_values = sp_subset[metric].dropna()

                if len(metric_values) > 0:
                    row_dict[f"sp{count}_{metric}_min"] = metric_values.min()
                    row_dict[f"sp{count}_{metric}_max"] = metric_values.max()
                    row_dict[f"sp{count}_{metric}_mean"] = metric_values.mean()
                    row_dict[f"sp{count}_{metric}_median"] = metric_values.median()

        for metric in metrics:
            metric_values = subset[metric].dropna()
            if len(metric_values) > 0:
                row_dict[f"total_{metric}_min"] = metric_values.min()
                row_dict[f"total_{metric}_max"] = metric_values.max()
                row_dict[f"total_{metric}_mean"] = metric_values.mean()
                row_dict[f"total_{metric}_median"] = metric_values.median()

        results.append(row_dict)

    return pd.DataFrame(results)


def plot_metric_vs_num_subproblems_by_ml_method(df, metric="Optimality_Gap%", output_path=None, filter_dict=None):
    try:
        df_plot = df[df["method_category"] == "ML"].copy()

        if filter_dict:
            for col, val in filter_dict.items():
                if val is not None and val != "total":
                    df_plot = df_plot[df_plot[col] == val]

        grouped = df_plot.groupby(["num_subproblems", "ml_method_type"])[metric].mean().reset_index()

        if len(grouped) == 0:
            print(f"No data for metric {metric}")
            return None

        if metric == "Time_Ratio":
            title = f"{metric} vs Number of Subproblems by ML Method"
            grouped = grouped.rename(columns={"ml_method_type": "combination"})
            _plot_time_ratio_layouts(
                grouped,
                axis_column="num_subproblems",
                title_base=title,
                output_path=output_path,
                legend_fontsize=None,
            )
            return None

        plt.figure(figsize=(10, 6))

        for method in grouped["ml_method_type"].unique():
            method_data = grouped[grouped["ml_method_type"] == method]
            method_data = method_data.sort_values("num_subproblems")
            plt.plot(
                method_data["num_subproblems"],
                method_data[metric],
                marker="o",
                label=method,
            )

        plt.xlabel("Number of Subproblems")
        plt.ylabel(metric)
        plt.title(f"{metric} vs Number of Subproblems by ML Method")
        plt.legend()

        plt.tight_layout()

        if output_path:
            output_path.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(output_path, dpi=300, bbox_inches="tight")
            print(f"Saved: {output_path}")

        plt.close()
        return None
    except Exception as e:
        print(f"Error plotting metric {metric}: {e}")
        plt.close()
        return None


def plot_metric_vs_num_subproblems_by_predictor(df, metric="Optimality_Gap%", output_path=None, filter_dict=None):
    try:
        df_plot = df[df["method_category"] == "ML"].copy()

        if filter_dict:
            for col, val in filter_dict.items():
                if val is not None and val != "total":
                    df_plot = df_plot[df_plot[col] == val]

        df_plot["combination"] = df_plot.apply(
            lambda row: combine_display_names(row["ml_method_type"], row["predictor_type"]),
            axis=1,
        )

        grouped = df_plot.groupby(["num_subproblems", "combination"])[metric].mean().reset_index()

        if len(grouped) == 0:
            print(f"No data for metric {metric}")
            return None

        if metric == "Time_Ratio":
            title = f"{metric} vs Number of Subproblems by ML Method + Predictor"
            _plot_time_ratio_layouts(
                grouped,
                axis_column="num_subproblems",
                title_base=title,
                output_path=output_path,
                legend_fontsize=8,
            )
            return None

        plt.figure(figsize=(12, 6))

        for combo in grouped["combination"].unique():
            combo_data = grouped[grouped["combination"] == combo]
            combo_data = combo_data.sort_values("num_subproblems")
            plt.plot(
                combo_data["num_subproblems"],
                combo_data[metric],
                marker="o",
                label=combo,
            )

        plt.xlabel("Number of Subproblems")
        plt.ylabel(metric)
        plt.title(f"{metric} vs Number of Subproblems by ML Method + Predictor")
        plt.legend(fontsize=8)

        plt.tight_layout()

        if output_path:
            output_path.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(output_path, dpi=300, bbox_inches="tight")
            print(f"Saved: {output_path}")

        plt.close()
        return None
    except Exception as e:
        print(f"Error plotting metric {metric}: {e}")
        plt.close()
        return None


def plot_metric_vs_num_subproblems_by_covariates(df, metric="Optimality_Gap%", output_path=None, filter_dict=None):
    try:
        df_plot = df[df["method_category"] == "ML"].copy()

        if filter_dict:
            for col, val in filter_dict.items():
                if val is not None and val != "total":
                    df_plot = df_plot[df_plot[col] == val]

        df_plot["combination"] = df_plot.apply(
            lambda row: combine_display_names(row["ml_method_type"], row["use_covariates"]),
            axis=1,
        )

        grouped = df_plot.groupby(["num_subproblems", "combination"])[metric].mean().reset_index()

        if len(grouped) == 0:
            print(f"No data for metric {metric}")
            return None

        if metric == "Time_Ratio":
            title = f"{metric} vs Number of Subproblems by ML Method + Covariates"
            _plot_time_ratio_layouts(
                grouped,
                axis_column="num_subproblems",
                title_base=title,
                output_path=output_path,
                legend_fontsize=8,
            )
            return None

        plt.figure(figsize=(12, 6))

        for combo in grouped["combination"].unique():
            combo_data = grouped[grouped["combination"] == combo]
            combo_data = combo_data.sort_values("num_subproblems")
            plt.plot(
                combo_data["num_subproblems"],
                combo_data[metric],
                marker="o",
                label=combo,
            )

        plt.xlabel("Number of Subproblems")
        plt.ylabel(metric)
        plt.title(f"{metric} vs Number of Subproblems by ML Method + Covariates")
        plt.legend(fontsize=8)

        plt.tight_layout()

        if output_path:
            output_path.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(output_path, dpi=300, bbox_inches="tight")
            print(f"Saved: {output_path}")

        plt.close()
        return None
    except Exception as e:
        print(f"Error plotting metric {metric}: {e}")
        plt.close()
        return None


def plot_metric_vs_num_subproblems_by_predictor_covariates(
    df, metric="Optimality_Gap%", output_path=None, filter_dict=None
):
    try:
        df_plot = df[df["method_category"] == "ML"].copy()

        if filter_dict:
            for col, val in filter_dict.items():
                if val is not None and val != "total":
                    df_plot = df_plot[df_plot[col] == val]

        df_plot["combination"] = df_plot.apply(
            lambda row: combine_display_names(
                row["ml_method_type"], row["predictor_type"], row["use_covariates"]
            ),
            axis=1,
        )

        grouped = df_plot.groupby(["num_subproblems", "combination"])[metric].mean().reset_index()

        if len(grouped) == 0:
            print(f"No data for metric {metric}")
            return None

        if metric == "Time_Ratio":
            title = f"{metric} vs Number of Subproblems by ML Method + Predictor + Covariates"
            _plot_time_ratio_layouts(
                grouped,
                axis_column="num_subproblems",
                title_base=title,
                output_path=output_path,
                legend_fontsize=8,
            )
            return None

        plt.figure(figsize=(12, 6))

        for combo in grouped["combination"].unique():
            combo_data = grouped[grouped["combination"] == combo]
            combo_data = combo_data.sort_values("num_subproblems")
            plt.plot(
                combo_data["num_subproblems"],
                combo_data[metric],
                marker="o",
                label=combo,
            )

        plt.xlabel("Number of Subproblems")
        plt.ylabel(metric)
        plt.title(f"{metric} vs Number of Subproblems by ML Method + Predictor + Covariates")
        plt.legend(fontsize=8)

        plt.tight_layout()

        if output_path:
            output_path.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(output_path, dpi=300, bbox_inches="tight")
            print(f"Saved: {output_path}")

        plt.close()
        return None
    except Exception as e:
        print(f"Error plotting metric {metric}: {e}")
        plt.close()
        return None

## Data Cleaning and Metric Helpers

Define directory, loading, filtering, duplicate removal, naming, splitting, and metric-computation utilities.


## Directory and File Helpers


In [60]:
def ensure_dirs(base_output_dir):
    base = Path(base_output_dir)
    figures_dir = base / "figures"
    tables_dir = base / "tables"

    base.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    tables_dir.mkdir(parents=True, exist_ok=True)

    return base, figures_dir, tables_dir



def ensure_nested_analysis_dirs(base_output_dir):
    """
    Create nested folder structure for analysis.
    Returns a dict mapping (data_group, problem_class, ml_callback_type) to folder paths.
    """
    base = Path(base_output_dir)
    base.mkdir(parents=True, exist_ok=True)
    nested_dirs = {}

    for data_group, _ in DATA_GROUPS_TO_ANALYZE:
        for callback_type in CALLBACK_TYPES_TO_ANALYZE:
            folder = base / data_group / "total" / callback_type
            folder.mkdir(parents=True, exist_ok=True)
            (folder / "tables").mkdir(parents=True, exist_ok=True)
            (folder / "figures" / "line_plots").mkdir(parents=True, exist_ok=True)
            (folder / "figures" / "boxplots").mkdir(parents=True, exist_ok=True)
            nested_dirs[(data_group, "total", callback_type)] = folder

    return nested_dirs


## Instance Name Cleaning Helpers


In [61]:
def read_input_file(file_path):
    suffix = Path(file_path).suffix.lower()
    if suffix in [".xlsx", ".xlsm", ".xls"]:
        return pd.read_excel(file_path)
    elif suffix == ".csv":
        return pd.read_csv(file_path)
    else:
        raise ValueError(f"Unsupported input format: {suffix}")



def _extract_final_instance_component(series, separators=("/", "\\")):
    values = series.astype(str)
    for separator in separators:
        values = values.str.rsplit(separator, n=1).str[-1]
    return values



def create_deterministic_instance(df):
    """
    Create 'deterministic_instance' column from 'instance_filename' by extracting
    the final path component. Handles both slash and backslash paths.
    Also create 'instance_filename_filile_name' for backward compatibility.
    """
    df = df.copy()
    df["deterministic_instance"] = _extract_final_instance_component(df["instance_filename"])
    df["instance_filename_filile_name"] = df["deterministic_instance"]
    return df



def add_determisitic_instance_name(df):
    """
    Append the requested 'determisitic_instance_name' column at the end.
    Values are extracted from the substring after the last '/' in instance_filename.
    """
    df = df.copy()
    if "determisitic_instance_name" in df.columns:
        df = df.drop(columns=["determisitic_instance_name"])
    df["determisitic_instance_name"] = df["instance_filename"].astype(str).str.rsplit("/", n=1).str[-1]
    return df



def filter_by_deterministic_instance(df, instances_list):
    """
    Filter dataframe to keep only rows where deterministic_instance 
    matches any value in instances_list.
    """
    df = df.copy()
    mask = df["deterministic_instance"].isin(instances_list)
    filtered = df[mask].copy()
    
    print(f"Original rows: {len(df)}")
    print(f"Filtered rows (matching deterministic_instance): {len(filtered)}")
    print(f"Rows removed: {len(df) - len(filtered)}")
    print(f"\nInstances included: {instances_list}")
    print(f"Unique instances in filtered data: {filtered['deterministic_instance'].unique().tolist()}")
    
    return filtered


## Duplicate Removal Helper


In [62]:
def remove_duplicates(df):
    """
    Remove duplicate experiment rows using the configured identity columns.

    DUPLICATE IDENTITY DEFINITION:
    Two rows are considered identical (duplicates) if they match on ALL columns
    in DUPLICATE_SUBSET. Null/NaN/empty values are treated as matching pairs:
    - If both rows have null in a column, they match on that column.
    - If both rows have the same empty string, they match on that column.
    - Duplicate groups can contain 2 or more identical rows.

    DUPLICATE SUBSET COLUMNS (20 columns):
    - problem_class
    - determisitic_instance
    - num_subproblems
    - method_category
    - ml_method_type
    - ml_callback_type
    - exact_callback_type
    - predictor_type
    - use_prediction
    - prediction_method
    - selected_subproblems_ratio
    - n_trials
    - m_trials
    - n_walks
    - walk_length
    - feature_dim
    - window
    - min_count
    - sg
    - use_covariates

    SELECTION RULE (how to choose representative from each duplicate group):
    Within each duplicate group, apply the following filters in order:
    
    1. TIME_LIMIT_SEC PREFERENCE (first filter):
       - Prefer rows with time_limit_sec == 10800
       - If none, prefer rows with time_limit_sec == 3600
       - If none, prefer rows with time_limit_sec as null/NaN
       - If all remaining rows failed to match any tier, use all remaining rows.
    
    2. MIPGAP_TARGET PREFERENCE (second filter):
       - From rows after step 1, prefer rows where:
         MIPGap_target is NOT null AND MIPGap_target != 0
       - If no such rows exist, use all rows from step 1.
    
    3. RANDOM SELECTION (final tie-breaker):
       - If multiple rows remain after steps 1-2, randomly select one.
       - Use DUPLICATE_RANDOM_STATE for reproducible results.
    
    RESULT:
    Returns a tuple of (deduplicated_df, removed_duplicates_df).
    """
    df = df.copy()

    missing_subset_cols = [col for col in DUPLICATE_SUBSET if col not in df.columns]
    duplicate_subset = [col for col in DUPLICATE_SUBSET if col in df.columns]

    if missing_subset_cols:
        warnings.warn(
            "The following duplicate-identity columns are missing and will be skipped: "
            + ", ".join(missing_subset_cols),
            RuntimeWarning,
        )

    if not duplicate_subset:
        warnings.warn(
            "No duplicate-identity columns are available. Skipping duplicate removal.",
            RuntimeWarning,
        )
        return df.reset_index(drop=True), pd.DataFrame(columns=df.columns)

    # Identify rows that have at least one exact duplicate (including itself).
    # dropna=False: null values are treated as equal and participate in grouping.
    duplicate_mask = df.duplicated(subset=duplicate_subset, keep=False)

    if not duplicate_mask.any():
        print(f"Duplicate identity columns used: {duplicate_subset}")
        print(f"No duplicate groups found.")
        return df.reset_index(drop=True), pd.DataFrame(columns=df.columns)

    kept_indices = []
    removed_indices = []

    # Process each duplicate group.
    for _, group_df in df[duplicate_mask].groupby(duplicate_subset, dropna=False, sort=False):
        candidate_df = group_df

        # STEP 1: Filter by time_limit_sec preference tier
        if "time_limit_sec" in candidate_df.columns:
            time_limit_numeric = pd.to_numeric(candidate_df["time_limit_sec"], errors="coerce")
            rows_10800 = candidate_df[time_limit_numeric == 10800]
            rows_3600 = candidate_df[time_limit_numeric == 3600]
            rows_null = candidate_df[time_limit_numeric.isna()]

            if not rows_10800.empty:
                candidate_df = rows_10800
            elif not rows_3600.empty:
                candidate_df = rows_3600
            elif len(rows_null) == len(candidate_df):
                candidate_df = rows_null

        # STEP 2: Filter by MIPGap_target preference
        if "MIPGap_target" in candidate_df.columns:
            mipgap_numeric = pd.to_numeric(candidate_df["MIPGap_target"], errors="coerce")
            preferred_mipgap_rows = candidate_df[mipgap_numeric.notna() & (mipgap_numeric != 0)]
            if not preferred_mipgap_rows.empty:
                candidate_df = preferred_mipgap_rows

        # STEP 3: Random selection from remaining candidates
        kept_idx = candidate_df.sample(n=1, random_state=DUPLICATE_RANDOM_STATE).index[0]
        kept_indices.append(kept_idx)
        removed_indices.extend([idx for idx in group_df.index if idx != kept_idx])

    unique_rows = df[~duplicate_mask]
    deduped_duplicate_rows = df.loc[kept_indices]
    deduped = (
        pd.concat([unique_rows, deduped_duplicate_rows])
        .sort_index()
        .reset_index(drop=True)
    )

    removed_duplicates = df.loc[removed_indices].copy().sort_index().reset_index(drop=True)

    print(f"Duplicate identity columns used: {duplicate_subset}")
    print(f"Duplicate groups found: {df[duplicate_mask].groupby(duplicate_subset, dropna=False).ngroups}")
    print(f"Rows removed as duplicates: {len(removed_duplicates)}")

    return deduped, removed_duplicates


## Short Name and Metric Helpers


In [63]:
def add_short_name_columns(df):
    """
    Add short-name columns for methods based on SHORT_NAME_MAP.
    """
    df = df.copy()
    
    # Create method_short from method_category
    df["method_short"] = df["method_category"].map(SHORT_NAME_MAP)
    
    # Create ml_method_short from ml_method_type
    df["ml_method_short"] = df["ml_method_type"].map(SHORT_NAME_MAP)
    
    # Preserve the raw predictor semantics in prediction_method so reporting can
    # apply the presentation map consistently.
    if "predictor_type" in df.columns:
        df["prediction_method"] = df["predictor_type"]
    
    return df



def compute_relative_metrics(df, metric_df=None):
    """
    Compute Optimality_Gap% and Time_Ratio metrics.

    For ML rows: match with an Exact row on the same instance_filename and compute
    metrics, regardless of the ML row's own stopping_reason. The caller controls
    which Exact rows are eligible by passing metric_df.

    For Exact rows: set reference metrics only when that Exact row's instance is
    present in metric_df, so non-eligible Exact baselines keep missing metrics.
    """
    df = df.copy()

    if metric_df is None:
        metric_df = df
    metric_df = metric_df.copy()

    exact_metric_df = metric_df[metric_df["method_category"] == "Exact"].copy()
    exact_metric_df["_instance_key"] = exact_metric_df["instance_filename"].astype(str)
    exact_metric_df = exact_metric_df.drop_duplicates(subset=["_instance_key"], keep="first")
    exact_by_instance = exact_metric_df.set_index("_instance_key")
    eligible_exact_instances = set(exact_by_instance.index)

    # Initialize metric columns
    df["exact_original_Prob_Obj_Value"] = np.nan
    df["exact_solution_time_sec"] = np.nan
    df["Optimality_Gap%"] = np.nan
    df["Time_Ratio"] = np.nan

    # For eligible Exact method rows, set reference values
    exact_mask = (df["method_category"] == "Exact") & df["instance_filename"].astype(str).isin(eligible_exact_instances)
    df.loc[exact_mask, "exact_original_Prob_Obj_Value"] = df.loc[exact_mask, "original_Prob_Obj_Value"]
    df.loc[exact_mask, "exact_solution_time_sec"] = df.loc[exact_mask, "solution_time_sec"]
    df.loc[exact_mask, "Optimality_Gap%"] = 0
    df.loc[exact_mask, "Time_Ratio"] = 1.0

    # For ML rows, find corresponding eligible Exact row and compute metrics
    ml_mask = df["method_category"] == "ML"
    ml_rows = df[ml_mask].copy()

    for idx in ml_rows.index:
        instance_filename = df.loc[idx, "instance_filename"]
        instance_key = str(instance_filename)

        if instance_key not in eligible_exact_instances:
            continue

        exact_row = exact_by_instance.loc[instance_key]
        exact_obj = exact_row["original_Prob_Obj_Value"]
        exact_time = exact_row["solution_time_sec"]

        df.loc[idx, "exact_original_Prob_Obj_Value"] = exact_obj
        df.loc[idx, "exact_solution_time_sec"] = exact_time

        ml_obj = df.loc[idx, "original_Prob_Obj_Value"]
        ml_time = df.loc[idx, "solution_time_sec"]

        if pd.notna(exact_obj) and exact_obj != 0 and pd.notna(ml_obj):
            df.loc[idx, "Optimality_Gap%"] = (ml_obj - exact_obj) / exact_obj

        if pd.notna(exact_time) and exact_time > 0 and pd.notna(ml_time):
            df.loc[idx, "Time_Ratio"] = ml_time / exact_time

    return df


## Exact Baseline Eligibility Filter

Use Exact rows to decide which instances have a valid baseline for downstream ML comparisons.


## Exact-Baseline Group Split Helper


In [64]:
def split_by_exact_baseline_optimality(df):
    """
    Split cleaned data into three Exact-baseline groups.

    group 1: all cleaned rows.
    group 2: all rows whose instance has an Exact OPTIMAL baseline.
    group 3: complement of group 2.

    The split is instance-level and based only on Exact rows. ML rows are included
    in group 2 whenever their corresponding Exact row is OPTIMAL, regardless of
    the ML row's own stopping_reason.
    """
    df_all = df.copy().reset_index(drop=True)

    exact_df = df_all[df_all["method_category"] == "Exact"].copy()
    if exact_df.empty:
        warnings.warn(
            "No Exact rows found. All rows will remain in cleaned_data and exact_nonoptimal_baseline.",
            RuntimeWarning,
        )
        empty_group = df_all.iloc[0:0].copy()
        return df_all, empty_group, df_all.copy(), set(), set(), set()

    exact_optimal_instances = set(
        exact_df.loc[exact_df["stopping_reason"] == "OPTIMAL", "instance_filename"]
        .dropna()
        .astype(str)
        .tolist()
    )
    exact_nonoptimal_instances = set(
        exact_df.loc[exact_df["stopping_reason"] != "OPTIMAL", "instance_filename"]
        .dropna()
        .astype(str)
        .tolist()
    )
    conflicting_instances = exact_optimal_instances & exact_nonoptimal_instances

    if conflicting_instances:
        warnings.warn(
            "Some instances have both OPTIMAL and non-OPTIMAL Exact rows after deduplication. "
            "They are assigned to exact_optimal_baseline because an OPTIMAL Exact baseline exists.",
            RuntimeWarning,
        )

    group2_instances = exact_optimal_instances
    group2_mask = df_all["instance_filename"].astype(str).isin(group2_instances)

    df_exact_optimal_baseline = df_all[group2_mask].copy().reset_index(drop=True)
    df_exact_nonoptimal_baseline = df_all[~group2_mask].copy().reset_index(drop=True)

    if len(df_exact_optimal_baseline) + len(df_exact_nonoptimal_baseline) != len(df_all):
        raise RuntimeError("Exact-baseline groups do not reconstruct cleaned_data.")

    print("\n" + "=" * 60)
    print("EXACT-BASELINE DATA GROUP SPLIT")
    print("=" * 60)
    print(f"Exact OPTIMAL instances discovered: {len(exact_optimal_instances)}")
    print(f"Exact non-OPTIMAL instances discovered: {len(exact_nonoptimal_instances)}")
    print(f"Instances with conflicting Exact statuses: {len(conflicting_instances)}")
    print(f"cleaned_data rows: {len(df_all)}")
    print(f"exact_optimal_baseline rows: {len(df_exact_optimal_baseline)}")
    print(f"exact_nonoptimal_baseline rows: {len(df_exact_nonoptimal_baseline)}")
    print(f"Group 2 + Group 3 rows: {len(df_exact_optimal_baseline) + len(df_exact_nonoptimal_baseline)}")

    return (
        df_all,
        df_exact_optimal_baseline,
        df_exact_nonoptimal_baseline,
        exact_optimal_instances,
        exact_nonoptimal_instances,
        conflicting_instances,
    )


## Analysis Method Filter Helper


In [65]:
def filter_to_analysis_methods(df):
    """Keep Exact rows plus the ML methods selected for metric analysis."""
    analysis_mask = (
        (df["method_category"] == "Exact")
        | (
            (df["method_category"] == "ML")
            & df["ml_method_type"].isin(ANALYSIS_ML_METHODS)
        )
    )
    predictor_mask = df["predictor_type"].isna() | (df["predictor_type"] != "KNN")
    return df[analysis_mask & predictor_mask].copy()



def split_by_stopping_reason_alignment(df):
    """
    Partition the dataset into two groups based on the baseline Exact stopping_reason.

    - Group 1 (all_optimal_baseline): instances where the Exact method achieved OPTIMAL.
    - Group 2 (not_all_optimal_baseline): all other instances.
    """
    df_all = df.copy().reset_index(drop=True)

    exact_df = df_all[df_all["method_category"] == "Exact"].copy()
    if exact_df.empty:
        warnings.warn(
            "No Exact rows found. Returning empty DataFrames for analysis groups.",
            RuntimeWarning,
        )
        df_all_optimal = pd.DataFrame(columns=df_all.columns)
        df_not_all_optimal = pd.DataFrame(columns=df_all.columns)
        return df_all_optimal, df_not_all_optimal

    exact_optimal_instances = set(
        exact_df.loc[exact_df["stopping_reason"] == "OPTIMAL", "instance_filename"]
        .dropna()
        .astype(str)
        .tolist()
    )

    df_all_optimal = df_all[
        df_all["instance_filename"].astype(str).isin(exact_optimal_instances)
    ].copy()
    df_not_all_optimal = df_all[
        ~df_all["instance_filename"].astype(str).isin(exact_optimal_instances)
    ].copy()

    print("\n" + "=" * 60)
    print("STOPPING-REASON ALIGNMENT DATA GROUP SPLIT")
    print("=" * 60)
    print(f"Exact OPTIMAL instances discovered: {len(exact_optimal_instances)}")
    print(f"all_optimal_baseline rows: {len(df_all_optimal)}")
    print(f"not_all_optimal_baseline rows: {len(df_not_all_optimal)}")

    return df_all_optimal, df_not_all_optimal


## Nonoptimal Percentage Table Helpers


In [66]:
def _add_method_for_nonoptimal_table(df):
    out = df.copy()
    out["_analysis_method"] = np.where(
        out["method_category"] == "Exact",
        "Exact",
        out["ml_method_type"],
    )
    return out


def build_nonoptimal_percentage_by_instance_table(df):
    """
    Build percentage of rows with stopping_reason != OPTIMAL by deterministic
    instance, num_subproblems, and method. Adds a final total row.
    """
    output_columns = ["determisitic_instance_name", "num_subproblems"] + NONOPTIMAL_PERCENT_METHOD_COLUMNS
    if df.empty:
        return pd.DataFrame(columns=output_columns)

    working = _add_method_for_nonoptimal_table(df)
    working = working[working["_analysis_method"].isin(NONOPTIMAL_PERCENT_METHOD_COLUMNS)].copy()
    if working.empty:
        return pd.DataFrame(columns=output_columns)

    working["_nonoptimal"] = working["stopping_reason"] != "OPTIMAL"

    grouped = (
        working
        .groupby(["determisitic_instance_name", "num_subproblems", "_analysis_method"], dropna=False)["_nonoptimal"]
        .mean()
        .mul(100)
        .reset_index(name="percent_nonoptimal")
    )
    table = grouped.pivot_table(
        index=["determisitic_instance_name", "num_subproblems"],
        columns="_analysis_method",
        values="percent_nonoptimal",
        aggfunc="first",
    ).reset_index()

    totals = (
        working
        .groupby("_analysis_method", dropna=False)["_nonoptimal"]
        .mean()
        .mul(100)
    )
    total_row = {"determisitic_instance_name": "Total", "num_subproblems": "Total"}
    for method in NONOPTIMAL_PERCENT_METHOD_COLUMNS:
        total_row[method] = totals.get(method, np.nan)

    for method in NONOPTIMAL_PERCENT_METHOD_COLUMNS:
        if method not in table.columns:
            table[method] = np.nan

    table = table[output_columns]
    table = table.sort_values(["determisitic_instance_name", "num_subproblems"], kind="stable")
    table = pd.concat([table, pd.DataFrame([total_row])], ignore_index=True)
    return table[output_columns]


def build_method_optimality_summary(df):
    """Build counts and OPTIMAL percentages for LABD predictor variants, RND, and Exact."""
    order = ["LABD + DT", "LABD + CLU", "LABD + kNN", "RND", "Exact"]
    output_columns = ["method_label", "count", "percent_optimal"]

    if df.empty:
        return pd.DataFrame([{col: np.nan for col in output_columns}]).iloc[0:0]

    working = apply_presentation_naming(df.copy())
    working["method_label"] = None

    if "method_category" in working.columns:
        exact_mask = working["method_category"] == "Exact"
        working.loc[exact_mask, "method_label"] = "Exact"

        ml_mask = working["method_category"] == "ML"
        working.loc[ml_mask & (working["ml_method_type"] == "RND"), "method_label"] = "RND"

        if "predictor_type" in working.columns:
            predictor_mask = working["ml_method_type"] == "LABD"
            predictors = {"DT", "CLU", "kNN"}
            for predictor in predictors:
                mask = predictor_mask & (working["predictor_type"] == predictor)
                working.loc[mask, "method_label"] = f"LABD + {predictor}"

    rows = []
    for label in order:
        subset = working[working["method_label"] == label]
        count = int(len(subset))
        if count > 0:
            percent_optimal = (subset["stopping_reason"] == "OPTIMAL").mean() * 100
        else:
            percent_optimal = np.nan
        rows.append({
            "method_label": label,
            "count": count,
            "percent_optimal": percent_optimal,
        })

    return pd.DataFrame(rows, columns=output_columns)


def build_method_optimality_by_instance(df):
    """Build method optimality summary per instance_filename."""
    if df.empty:
        return pd.DataFrame(columns=["instance_filename", "method_label", "count", "percent_optimal"])

    rows = []
    for instance in sorted(df["instance_filename"].dropna().astype(str).unique()):
        subset = df[df["instance_filename"].astype(str) == instance].copy()
        summary = build_method_optimality_summary(subset)
        if summary.empty:
            continue
        summary.insert(0, "instance_filename", instance)
        rows.append(summary)

    if not rows:
        return pd.DataFrame(columns=["instance_filename", "method_label", "count", "percent_optimal"])

    return pd.concat(rows, ignore_index=True)


## Boxplot Helpers

Create metric distribution plots by method, predictor, covariate setting, and split value.


## Boxplot Shared Helpers


In [67]:
def _format_split_value_for_filename(value):
    text = str(value).replace(".", "p").replace("-", "m")
    return text


def _format_ratio_percent(value):
    if pd.isna(value):
        return ""
    percent = value * 100
    if abs(percent - round(percent)) < 1e-6:
        return str(int(round(percent)))
    return str(round(percent, 2)).replace(".", "p")


def _format_split_label(split_column, value):
    if split_column == "num_subproblems":
        return f"sp{int(value)}"
    if split_column == "selected_subproblems_ratio":
        percent = value * 100
        if abs(percent - round(percent)) < 1e-6:
            percent_text = f"{int(round(percent))}%"
        else:
            percent_text = f"{percent:.2f}%"
        return f"Exact Subproblems Ratio {percent_text}"
    return f"{split_column} {value}"


def _format_split_suffix(split_column, split_prefix, value):
    if split_column == "num_subproblems":
        return f"{split_prefix}{int(value)}"
    if split_column == "selected_subproblems_ratio":
        return f"{split_prefix}{_format_ratio_percent(value)}"
    return f"{split_prefix}{_format_split_value_for_filename(value)}"


## Boxplots by ML Method


In [68]:
def plot_boxplots_by_ml_method(df, metric="Optimality_Gap%", output_path=None, filter_dict=None, split_column="num_subproblems", split_prefix="sp"):
    """
    Create boxplots showing distribution of metric for each ml_method_type.
    Saves separate files for: total, sp100, sp300, sp500 (whatever subproblem counts exist)
    output_path should be like: /path/to/boxplot_gap_by_ml_method
    Files will be saved as: boxplot_gap_by_ml_method_total.png, boxplot_gap_by_ml_method_sp100.png, etc.
    """
    try:
        df_plot = df[df["method_category"] == "ML"].copy()

        if filter_dict:
            for col, val in filter_dict.items():
                if val is not None and val != "total":
                    df_plot = df_plot[df_plot[col] == val]

        # Get unique values for the split axis
        split_values = sorted([x for x in df_plot[split_column].unique() if pd.notna(x)])

        # Convert output_path to string and remove .png if present
        if output_path:
            output_path = str(output_path).replace('.png', '')
            output_dir = Path(output_path).parent
            output_dir.mkdir(parents=True, exist_ok=True)

        # Plot total
        fig, ax = plt.subplots(figsize=(10, 6))
        data_for_box = [df_plot[df_plot["ml_method_type"] == method][metric].dropna().values
                        for method in sorted(df_plot["ml_method_type"].unique())]
        labels = sorted(df_plot["ml_method_type"].unique())

        if len(data_for_box) > 0 and any(len(d) > 0 for d in data_for_box):
            bp = ax.boxplot(data_for_box, tick_labels=labels)
            ax.set_title(f"{metric_display_name(metric)} by ML Method (Total)")
            ax.set_ylabel(metric)
            ax.tick_params(axis='x', rotation=45)

        plt.tight_layout()
        if output_path:
            plt.savefig(f"{output_path}_total.png", dpi=300, bbox_inches="tight")
        plt.close()

        # Plot for each split-axis value
        for split_value in split_values:
            fig, ax = plt.subplots(figsize=(10, 6))
            df_sp = df_plot[df_plot[split_column] == split_value]

            data_for_box = [df_sp[df_sp["ml_method_type"] == method][metric].dropna().values
                            for method in sorted(df_sp["ml_method_type"].unique())]
            labels = sorted(df_sp["ml_method_type"].unique())

            if len(data_for_box) > 0 and any(len(d) > 0 for d in data_for_box):
                bp = ax.boxplot(data_for_box, tick_labels=labels)
                ax.set_title(f"{metric_display_name(metric)} by ML Method ({_format_split_label(split_column, split_value)})")
                ax.set_ylabel(metric)
                ax.tick_params(axis='x', rotation=45)

            plt.tight_layout()
            if output_path:
                plt.savefig(f"{output_path}_{_format_split_suffix(split_column, split_prefix, split_value)}.png", dpi=300, bbox_inches="tight")
            plt.close()

        return None
    except Exception as e:
        print(f"Error creating boxplot for metric {metric}: {e}")
        plt.close()
        return None

## Boxplots by Predictor


In [69]:
def plot_boxplots_by_predictor(df, metric="Optimality_Gap%", output_path=None, filter_dict=None, split_column="num_subproblems", split_prefix="sp"):
    """
    Create boxplots showing distribution of metric for each (ml_method_type, predictor_type) combination.
    Saves separate files for: total, sp100, sp300, sp500 (whatever subproblem counts exist)
    output_path should be like: /path/to/boxplot_gap_by_predictor
    Files will be saved as: boxplot_gap_by_predictor_total.png, boxplot_gap_by_predictor_sp100.png, etc.
    """
    try:
        df_plot = df[df["method_category"] == "ML"].copy()

        if filter_dict:
            for col, val in filter_dict.items():
                if val is not None and val != "total":
                    df_plot = df_plot[df_plot[col] == val]

        df_plot["combination"] = df_plot.apply(
            lambda row: combine_display_names(row["ml_method_type"], row["predictor_type"]),
            axis=1,
        )

        # Get unique values for the split axis
        split_values = sorted([x for x in df_plot[split_column].unique() if pd.notna(x)])

        # Convert output_path to string and remove .png if present
        if output_path:
            output_path = str(output_path).replace('.png', '')
            output_dir = Path(output_path).parent
            output_dir.mkdir(parents=True, exist_ok=True)

        # Plot total
        fig, ax = plt.subplots(figsize=(14, 6))
        all_combos = sorted(df_plot["combination"].unique())
        data_for_box = [df_plot[df_plot["combination"] == combo][metric].dropna().values
                        for combo in all_combos]

        if len(data_for_box) > 0 and any(len(d) > 0 for d in data_for_box):
            bp = ax.boxplot(data_for_box, tick_labels=all_combos)
            ax.set_title(f"{metric_display_name(metric)} by ML Method + Predictor (Total)")
            ax.set_ylabel(metric)
            ax.tick_params(axis='x', rotation=45)

        plt.tight_layout()
        if output_path:
            plt.savefig(f"{output_path}_total.png", dpi=300, bbox_inches="tight")
        plt.close()

        # Plot for each split-axis value
        for split_value in split_values:
            fig, ax = plt.subplots(figsize=(14, 6))
            df_sp = df_plot[df_plot[split_column] == split_value]

            all_combos = sorted(df_sp["combination"].unique())
            data_for_box = [df_sp[df_sp["combination"] == combo][metric].dropna().values
                            for combo in all_combos]

            if len(data_for_box) > 0 and any(len(d) > 0 for d in data_for_box):
                bp = ax.boxplot(data_for_box, tick_labels=all_combos)
                ax.set_title(f"{metric_display_name(metric)} by ML Method + Predictor ({_format_split_label(split_column, split_value)})")
                ax.set_ylabel(metric)
                ax.tick_params(axis='x', rotation=45)

            plt.tight_layout()
            if output_path:
                plt.savefig(f"{output_path}_{_format_split_suffix(split_column, split_prefix, split_value)}.png", dpi=300, bbox_inches="tight")
            plt.close()

        return None
    except Exception as e:
        print(f"Error creating boxplot for metric {metric}: {e}")
        plt.close()
        return None

## Boxplots by Covariates


In [70]:
def plot_boxplots_by_covariates(df, metric="Optimality_Gap%", output_path=None, filter_dict=None, split_column="num_subproblems", split_prefix="sp"):
    """
    Create boxplots showing distribution of metric for each (ml_method_type, use_covariates) combination.
    Saves separate files for: total, sp100, sp300, sp500 (whatever subproblem counts exist)
    output_path should be like: /path/to/boxplot_gap_by_covariates
    Files will be saved as: boxplot_gap_by_covariates_total.png, boxplot_gap_by_covariates_sp100.png, etc.
    """
    try:
        df_plot = df[df["method_category"] == "ML"].copy()

        if filter_dict:
            for col, val in filter_dict.items():
                if val is not None and val != "total":
                    df_plot = df_plot[df_plot[col] == val]

        df_plot["combination"] = df_plot.apply(
            lambda row: combine_display_names(row["ml_method_type"], row["use_covariates"]),
            axis=1,
        )

        # Get unique values for the split axis
        split_values = sorted([x for x in df_plot[split_column].unique() if pd.notna(x)])

        # Convert output_path to string and remove .png if present
        if output_path:
            output_path = str(output_path).replace('.png', '')
            output_dir = Path(output_path).parent
            output_dir.mkdir(parents=True, exist_ok=True)

        # Plot total
        fig, ax = plt.subplots(figsize=(12, 6))
        all_combos = sorted(df_plot["combination"].unique())
        data_for_box = [df_plot[df_plot["combination"] == combo][metric].dropna().values
                        for combo in all_combos]

        if len(data_for_box) > 0 and any(len(d) > 0 for d in data_for_box):
            bp = ax.boxplot(data_for_box, tick_labels=all_combos)
            ax.set_title(f"{metric_display_name(metric)} by ML Method + Covariates (Total)")
            ax.set_ylabel(metric)
            ax.tick_params(axis='x', rotation=45)

        plt.tight_layout()
        if output_path:
            plt.savefig(f"{output_path}_total.png", dpi=300, bbox_inches="tight")
        plt.close()

        # Plot for each split-axis value
        for split_value in split_values:
            fig, ax = plt.subplots(figsize=(12, 6))
            df_sp = df_plot[df_plot[split_column] == split_value]

            all_combos = sorted(df_sp["combination"].unique())
            data_for_box = [df_sp[df_sp["combination"] == combo][metric].dropna().values
                            for combo in all_combos]

            if len(data_for_box) > 0 and any(len(d) > 0 for d in data_for_box):
                bp = ax.boxplot(data_for_box, tick_labels=all_combos)
                ax.set_title(f"{metric_display_name(metric)} by ML Method + Covariates ({_format_split_label(split_column, split_value)})")
                ax.set_ylabel(metric)
                ax.tick_params(axis='x', rotation=45)

            plt.tight_layout()
            if output_path:
                plt.savefig(f"{output_path}_{_format_split_suffix(split_column, split_prefix, split_value)}.png", dpi=300, bbox_inches="tight")
            plt.close()

        return None
    except Exception as e:
        print(f"Error creating boxplot for metric {metric}: {e}")
        plt.close()
        return None

## Load Raw Data

Read the aggregate experiment file and inspect its raw shape and columns.


In [71]:

# =========================
# STEP 1: Load and deduplicate
# =========================
df = read_input_file(INPUT_FILE)

print("Shape of raw input data:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst few rows:")
df.head()


Shape of raw input data: (8747, 90)
Columns: ['problem_class', 'instance_filename', 'num_subproblems', 'num_first_stage_variables', 'method_category', 'ml_method_type', 'ml_callback_type', 'exact_callback_type', 'predictor_type', 'use_prediction', 'prediction_method', 'dual_difference_pred_loss', 'obj_difference_pred_loss', 'fallback_method', 'switching_strategy', 'KNN_k_neighbors', 'knn_find_optimal_k', 'tree_min_impurity_decrease', 'tree_quantile_grid_size', 'clustering_random_state', 'clustering_init_method', 'clustering_max_iter', 'clustering_tol', 'selected_subproblems_ratio', 'n_trials', 'm_trials', 'subproblem_selection_method', 'scenario_generation_method', 'scenario_info_dict', 'n_walks', 'walk_length', 'feature_dim', 'window', 'min_count', 'sg', 'seed', 'objective_value', 'original_Prob_Obj_Value', 'first_stage_optimnal_values', 'solution_time_sec', 'feature_gen_time_sec', 'unified_selection_training_time_sec', 'phase0_execution_time_sec', 'phase0_solution_generation_time_sec

,problem_class,instance_filename,num_subproblems,num_first_stage_variables,method_category,ml_method_type,ml_callback_type,exact_callback_type,predictor_type,use_prediction,...,mipnode_root,mipnode_nonroot,avg_mipsol_per_nonroot_node,avg_mipnode_per_nonroot_node,num_solutions_reevaluated,max_objective_drift,mean_objective_drift,std_objective_drift,MIPGap_target,MIPGap_Stop
0,SCFLP,SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...,100,10,Exact,NaN,NaN,int_and_LP,NaN,False,...,4,417,0.115914,0.819253,76,4.398723e+07,1.953552e+07,0.0,NaN,NaN
1,SCFLP,SCFLP/scenarios_s300_sigma0.1_0.3_cov6_merve_w...,300,10,Exact,NaN,NaN,int_and_LP,NaN,False,...,3,333,0.124473,0.702532,77,1.324193e+08,5.900273e+07,0.0,NaN,NaN
2,SCFLP,SCFLP/scenarios_s500_sigma0.1_0.3_cov6_merve_w...,500,10,Exact,NaN,NaN,int_and_LP,NaN,False,...,3,324,0.150901,0.729730,78,2.211981e+08,9.866574e+07,0.0,NaN,NaN
3,SCFLP,SCFLP/scenarios_s700_sigma0.1_0.3_cov6_merve_w...,700,10,Exact,NaN,NaN,int_and_LP,NaN,False,...,3,277,0.132316,0.704835,66,3.086905e+08,1.399909e+08,0.0,NaN,NaN
4,SCFLP,SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...,100,10,ML,ML_Augmented_BD,ML_at_integer_exact_at_LP,NaN,TREE,True,...,3,524,0.127527,0.814930,96,4.776343e+07,1.995073e+07,0.0,NaN,NaN


## Derive and Filter Instance Names

Create deterministic instance names and retain only the configured experiment instances.


In [72]:

# =========================
# STEP 2: Create deterministic_instance and filter
# =========================
df = create_deterministic_instance(df)

print("\nDeterministic instance column created.")
print("Sample values:")
print(df[["instance_filename", "deterministic_instance"]].head(10))

print("\n" + "="*60)
print("FILTERING BY DETERMINISTIC INSTANCE")
print("="*60)
df = filter_by_deterministic_instance(df, INSTANCES_TO_INCLUDE)

print("\n" + "="*60)
print("EXCLUDING ML METHOD TYPE: ML_at_integer_exact_at_LP")
print("="*60)
if "ml_method_type" in df.columns:
    before_rows = len(df)
    df = df[df["ml_method_type"] != "ML_at_integer_exact_at_LP"].copy()
    print(f"Rows removed: {before_rows - len(df)}")
else:
    print("Column ml_method_type not found; no rows removed.")



Deterministic instance column created.
Sample values:
                                   instance_filename deterministic_instance
0  SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...      p4_stochastic.txt
1  SCFLP/scenarios_s300_sigma0.1_0.3_cov6_merve_w...      p4_stochastic.txt
2  SCFLP/scenarios_s500_sigma0.1_0.3_cov6_merve_w...      p4_stochastic.txt
3  SCFLP/scenarios_s700_sigma0.1_0.3_cov6_merve_w...      p4_stochastic.txt
4  SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...      p4_stochastic.txt
5  SCFLP/scenarios_s300_sigma0.1_0.3_cov6_merve_w...      p4_stochastic.txt
6  SCFLP/scenarios_s500_sigma0.1_0.3_cov6_merve_w...      p4_stochastic.txt
7  SCFLP/scenarios_s700_sigma0.1_0.3_cov6_merve_w...      p4_stochastic.txt
8  SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...      p4_stochastic.txt
9  SCFLP/scenarios_s300_sigma0.1_0.3_cov6_merve_w...      p4_stochastic.txt

FILTERING BY DETERMINISTIC INSTANCE
Original rows: 8747
Filtered rows (matching deterministic_instance): 874

## Remove Duplicate Runs and Export CSV

Apply the two-stage duplicate rule and write the deduplicated dataset to CSV.


In [73]:

# =========================
# STEP 3: Remove duplicates
# =========================
print("\n" + "="*60)
print("REMOVING DUPLICATES")
print("="*60)
df_deduped, removed_duplicates = remove_duplicates(df)

print(f"\nDuplicate removal complete.")

output_deduped_csv_path = Path(OUTPUT_DIR) / OUTPUT_DEDUPED_CSV_NAME
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
df_deduped.to_csv(output_deduped_csv_path, index=False)
print(f"Deduplicated data saved to CSV: {output_deduped_csv_path}")

if not removed_duplicates.empty:
    print(f"\nFirst few removed duplicates:")
    removed_duplicates.head()



REMOVING DUPLICATES
Duplicate identity columns used: ['problem_class', 'deterministic_instance', 'num_subproblems', 'method_category', 'ml_method_type', 'ml_callback_type', 'exact_callback_type', 'predictor_type', 'use_prediction', 'prediction_method', 'selected_subproblems_ratio', 'n_trials', 'm_trials', 'n_walks', 'walk_length', 'feature_dim', 'window', 'min_count', 'sg', 'use_covariates']
Duplicate groups found: 2204
Rows removed as duplicates: 4543

Duplicate removal complete.
Deduplicated data saved to CSV: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/deduplicated_experiment_data.csv

First few removed duplicates:


## Filter Eligible Baselines and Split Data

Remove instances without an Exact OPTIMAL baseline, add short names, and split by stopping reason.


In [74]:

# =========================
# STEP 4: Add short names and split by Exact-baseline status
# =========================
print("\n" + "="*60)
print("ADDING SHORT NAMES")
print("="*60)
df_deduped = add_short_name_columns(df_deduped)
df_deduped = add_determisitic_instance_name(df_deduped)

print("\nShort names added.")
print(df_deduped[["method_category", "ml_method_type", "method_short", "ml_method_short"]].head(10))

print("\n" + "="*60)
print("SPLITTING BY EXACT-BASELINE STATUS")
print("="*60)
(
    df_cleaned_data,
    df_exact_optimal_baseline,
    df_exact_nonoptimal_baseline,
    exact_optimal_instances,
    exact_nonoptimal_instances,
    conflicting_exact_status_instances,
) = split_by_exact_baseline_optimality(df_deduped)

# Backward-compatible aliases for workbook/export naming used later in the notebook.
df_all = df_cleaned_data
df_group2 = df_exact_optimal_baseline
df_group3 = df_exact_nonoptimal_baseline



ADDING SHORT NAMES

Short names added.
  method_category   ml_method_type method_short ml_method_short
0           Exact              NaN        exact             NaN
1           Exact              NaN        exact             NaN
2           Exact              NaN        exact             NaN
3           Exact              NaN        exact             NaN
4              ML  ML_Augmented_BD          NaN          ml_ftr
5              ML  ML_Augmented_BD          NaN          ml_ftr
6              ML  ML_Augmented_BD          NaN          ml_ftr
7              ML  ML_Augmented_BD          NaN          ml_ftr
8              ML  ML_Augmented_BD          NaN          ml_ftr
9              ML  ML_Augmented_BD          NaN          ml_ftr

SPLITTING BY EXACT-BASELINE STATUS

EXACT-BASELINE DATA GROUP SPLIT
Exact OPTIMAL instances discovered: 20
Exact non-OPTIMAL instances discovered: 24
Instances with conflicting Exact statuses: 0
cleaned_data rows: 4204
exact_optimal_baseline rows: 2357
ex

## Compute Relative Metrics

Compute `Optimality_Gap%` and `Time_Ratio` using Exact OPTIMAL baseline rows as the metric reference. ML rows in the Exact OPTIMAL baseline group are included regardless of their own `stopping_reason`.

In [75]:

# =========================
# STEP 5: Compute metrics using Exact OPTIMAL baselines
# =========================
print("\n" + "="*60)
print("COMPUTING METRICS (Exact OPTIMAL baselines as reference)")
print("="*60)

exact_metric_reference = df_exact_optimal_baseline[
    (df_exact_optimal_baseline["method_category"] == "Exact")
    & (df_exact_optimal_baseline["stopping_reason"] == "OPTIMAL")
].copy()

df_cleaned_data = compute_relative_metrics(df_cleaned_data, metric_df=exact_metric_reference)
df_exact_optimal_baseline = compute_relative_metrics(df_exact_optimal_baseline, metric_df=exact_metric_reference)
df_exact_nonoptimal_baseline = compute_relative_metrics(df_exact_nonoptimal_baseline, metric_df=exact_metric_reference)

# Keep the requested derived name column as the final exported column.
df_cleaned_data = add_determisitic_instance_name(df_cleaned_data)
df_exact_optimal_baseline = add_determisitic_instance_name(df_exact_optimal_baseline)
df_exact_nonoptimal_baseline = add_determisitic_instance_name(df_exact_nonoptimal_baseline)

# Keep aliases aligned after metric computation.
df_all = df_cleaned_data
df_group2 = df_exact_optimal_baseline
df_group3 = df_exact_nonoptimal_baseline

print("\nMetrics computed.")
print(df_exact_optimal_baseline[
    [
        "instance_filename",
        "determisitic_instance_name",
        "method_category",
        "ml_method_type",
        "stopping_reason",
        "original_Prob_Obj_Value",
        "solution_time_sec",
        "exact_original_Prob_Obj_Value",
        "exact_solution_time_sec",
        "Optimality_Gap%",
        "Time_Ratio",
    ]
].head(20))



COMPUTING METRICS (Exact OPTIMAL baselines as reference)

Metrics computed.
                                    instance_filename  \
0   SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...   
1   SCFLP/scenarios_s300_sigma0.1_0.3_cov6_merve_w...   
2   SCFLP/scenarios_s500_sigma0.1_0.3_cov6_merve_w...   
3   SCFLP/scenarios_s700_sigma0.1_0.3_cov6_merve_w...   
4   SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...   
5   SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...   
6   SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...   
7   SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...   
8   SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...   
9   SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...   
10  SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...   
11  SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...   
12  SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...   
13  SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...   
14  SCFLP/scenarios_s100_sigma0.1_0.3_cov6_merve_w...   
15  SCFLP/s

## Diagnostic Checks

Report metric coverage and the available problem, method, and callback categories.


In [76]:

# =========================
# STEP 6: Diagnostic checks
# =========================
print("\n" + "="*60)
print("DIAGNOSTIC CHECKS")
print("="*60)

ml_with_metrics = df_exact_optimal_baseline[
    (df_exact_optimal_baseline["method_category"] == "ML")
    & (df_exact_optimal_baseline["Optimality_Gap%"].notna())
]
ml_rows_group2 = df_exact_optimal_baseline[df_exact_optimal_baseline["method_category"] == "ML"]

print(f"\nML rows in exact_optimal_baseline with computed metrics: {len(ml_with_metrics)}")
print(f"ML rows in exact_optimal_baseline without matching Exact: {len(ml_rows_group2) - len(ml_with_metrics)}")

print(f"\nGroup disjoint/completeness check:")
print(f"  cleaned_data rows: {len(df_cleaned_data)}")
print(f"  exact_optimal_baseline + exact_nonoptimal_baseline rows: {len(df_exact_optimal_baseline) + len(df_exact_nonoptimal_baseline)}")

for group_name, group_df in [
    ("cleaned_data", df_cleaned_data),
    ("exact_optimal_baseline", df_exact_optimal_baseline),
    ("exact_nonoptimal_baseline", df_exact_nonoptimal_baseline),
]:
    print(f"\n{group_name}:")
    print(f"  Rows: {len(group_df)}")
    print(f"  Unique problem classes: {group_df['problem_class'].nunique()}")
    print(f"  ML methods: {group_df[group_df['method_category'] == 'ML']['ml_method_type'].dropna().unique().tolist()}")
    print(f"  ML callback types: {group_df[group_df['method_category'] == 'ML']['ml_callback_type'].dropna().unique().tolist()}")



DIAGNOSTIC CHECKS

ML rows in exact_optimal_baseline with computed metrics: 2335
ML rows in exact_optimal_baseline without matching Exact: 2

Group disjoint/completeness check:
  cleaned_data rows: 4204
  exact_optimal_baseline + exact_nonoptimal_baseline rows: 4204

cleaned_data:
  Rows: 4204
  Unique problem classes: 2
  ML methods: ['ML_Augmented_BD', 'ML_Augmented_BD_with_duals', 'Uncertainty_Similarity', 'Random_Sampling']
  ML callback types: ['ML_at_integer_exact_at_LP', 'ML_at_both', 'ML_at_LP_exact_at_integer']

exact_optimal_baseline:
  Rows: 2357
  Unique problem classes: 2
  ML methods: ['ML_Augmented_BD', 'ML_Augmented_BD_with_duals', 'Uncertainty_Similarity', 'Random_Sampling']
  ML callback types: ['ML_at_integer_exact_at_LP', 'ML_at_both', 'ML_at_LP_exact_at_integer']

exact_nonoptimal_baseline:
  Rows: 1847
  Unique problem classes: 2
  ML methods: ['ML_Augmented_BD_with_duals', 'Uncertainty_Similarity', 'ML_Augmented_BD', 'Random_Sampling']
  ML callback types: ['ML_

# =========================
# STEP 7: Generate analysis for nested folder structure
# =========================
print("\n" + "="*60)
print("GENERATING HIERARCHICAL ANALYSIS")
print("="*60)

base_dir, figures_dir, tables_dir = ensure_dirs(OUTPUT_DIR)
print(f"\nBase output directory: {base_dir}")

# Create All_Optimal and NOT_all_Optimal groups based on stopping_reason alignment
df_all_optimal, df_not_all_optimal = split_by_stopping_reason_alignment(df_cleaned_data)

# Apply num_subproblems filter (100, 300, 500 only) to all groups
NUM_SUBPROBLEMS_FILTER = [100, 300, 500]

print("\n" + "="*60)
print("APPLYING NUM_SUBPROBLEMS FILTER")
print("="*60)
print(f"Filtering to num_subproblems in {NUM_SUBPROBLEMS_FILTER}")

analysis_data_groups = {
    "cleaned_data": df_cleaned_data[df_cleaned_data["num_subproblems"].isin(NUM_SUBPROBLEMS_FILTER)].copy(),
    "exact_optimal_baseline": df_exact_optimal_baseline[df_exact_optimal_baseline["num_subproblems"].isin(NUM_SUBPROBLEMS_FILTER)].copy(),
    "exact_nonoptimal_baseline": df_exact_nonoptimal_baseline[df_exact_nonoptimal_baseline["num_subproblems"].isin(NUM_SUBPROBLEMS_FILTER)].copy(),
    "All_Optimal": df_all_optimal[df_all_optimal["num_subproblems"].isin(NUM_SUBPROBLEMS_FILTER)].copy(),
    "NOT_all_Optimal": df_not_all_optimal[df_not_all_optimal["num_subproblems"].isin(NUM_SUBPROBLEMS_FILTER)].copy(),
}

observed_callback_types = sorted({
    x
    for group_df in analysis_data_groups.values()
    for x in group_df[group_df["method_category"] == "ML"]["ml_callback_type"].dropna().unique().tolist()
})
callback_types = CALLBACK_TYPES_TO_ANALYZE + [
    x for x in observed_callback_types if x not in CALLBACK_TYPES_TO_ANALYZE
]

print(f"\nData groups to analyze: {list(analysis_data_groups.keys())}")
print(f"ML methods kept for metric analysis: {ANALYSIS_ML_METHODS}")
print(f"Callback types to analyze: {callback_types}")

analysis_results = {name: df.copy() for name, df in analysis_data_groups.items()}

analysis_filtered_results = {
    name: filter_to_analysis_methods(group_df)
    for name, group_df in analysis_data_groups.items()
}

analysis_filtered_dir = Path(OUTPUT_DIR) / "analysis_filtered"
analysis_filtered_dir.mkdir(parents=True, exist_ok=True)
for group_name, group_df in analysis_filtered_results.items():
    output_path = analysis_filtered_dir / f"{group_name}_analysis_filtered.csv"
    group_df.to_csv(output_path, index=False)
    print(f"Exported {group_name} analysis-filtered data: {output_path}")

cleaned_data_analysis = filter_to_analysis_methods(df_cleaned_data)
cleaned_data_analysis = cleaned_data_analysis[
    cleaned_data_analysis["num_subproblems"].isin(NUM_SUBPROBLEMS_FILTER)
].copy()
if "ml_callback_type" in cleaned_data_analysis.columns:
    cleaned_data_analysis = cleaned_data_analysis[
        cleaned_data_analysis["ml_callback_type"] != "ML_at_LP_exact_at_integer"
    ].copy()

cleaned_data_analysis_path = Path(OUTPUT_DIR) / "cleaned_data_analysis.xlsx"
with pd.ExcelWriter(cleaned_data_analysis_path, engine="openpyxl") as writer:
    cleaned_data_analysis.to_excel(writer, sheet_name="cleaned_data_analysis", index=False)
print(f"Exported cleaned_data_analysis: {cleaned_data_analysis_path}")

print(f"\nData splits prepared (after num_subproblems filter):") 
for group_name, group_df in analysis_results.items():
    print(f"  - {group_name}: {len(group_df)} rows")


In [77]:
# =========================
# STEP 7: Generate analysis for nested folder structure
# =========================
print("\n" + "="*60)
print("GENERATING HIERARCHICAL ANALYSIS")
print("="*60)

base_dir, figures_dir, tables_dir = ensure_dirs(OUTPUT_DIR)
print(f"\nBase output directory: {base_dir}")

# Create All_Optimal and NOT_all_Optimal groups based on stopping_reason alignment
df_all_optimal, df_not_all_optimal = split_by_stopping_reason_alignment(df_cleaned_data)

# Apply num_subproblems filter (100, 300, 500 only) to all groups
NUM_SUBPROBLEMS_FILTER = [100, 300, 500]

print("\n" + "="*60)
print("APPLYING NUM_SUBPROBLEMS FILTER")
print("="*60)
print(f"Filtering to num_subproblems in {NUM_SUBPROBLEMS_FILTER}")

analysis_data_groups = {
    "cleaned_data": df_cleaned_data[df_cleaned_data["num_subproblems"].isin(NUM_SUBPROBLEMS_FILTER)].copy(),
    "exact_optimal_baseline": df_exact_optimal_baseline[df_exact_optimal_baseline["num_subproblems"].isin(NUM_SUBPROBLEMS_FILTER)].copy(),
    "exact_nonoptimal_baseline": df_exact_nonoptimal_baseline[df_exact_nonoptimal_baseline["num_subproblems"].isin(NUM_SUBPROBLEMS_FILTER)].copy(),
    "All_Optimal": df_all_optimal[df_all_optimal["num_subproblems"].isin(NUM_SUBPROBLEMS_FILTER)].copy(),
    "NOT_all_Optimal": df_not_all_optimal[df_not_all_optimal["num_subproblems"].isin(NUM_SUBPROBLEMS_FILTER)].copy(),
}

observed_callback_types = sorted({
    x
    for group_df in analysis_data_groups.values()
    for x in group_df[group_df["method_category"] == "ML"]["ml_callback_type"].dropna().unique().tolist()
})
callback_types = CALLBACK_TYPES_TO_ANALYZE + [
    x for x in observed_callback_types if x not in CALLBACK_TYPES_TO_ANALYZE
]

print(f"\nData groups to analyze: {list(analysis_data_groups.keys())}")
print(f"ML methods kept for metric analysis: {ANALYSIS_ML_METHODS}")
print(f"Callback types to analyze: {callback_types}")

analysis_results = {name: df.copy() for name, df in analysis_data_groups.items()}

analysis_filtered_results = {
    name: filter_to_analysis_methods(group_df)
    for name, group_df in analysis_data_groups.items()
}

analysis_filtered_dir = Path(OUTPUT_DIR) / "analysis_filtered"
analysis_filtered_dir.mkdir(parents=True, exist_ok=True)
for group_name, group_df in analysis_filtered_results.items():
    output_path = analysis_filtered_dir / f"{group_name}_analysis_filtered.csv"
    group_df.to_csv(output_path, index=False)
    print(f"Exported {group_name} analysis-filtered data: {output_path}")

cleaned_data_analysis = filter_to_analysis_methods(df_cleaned_data)
cleaned_data_analysis = cleaned_data_analysis[
    cleaned_data_analysis["num_subproblems"].isin(NUM_SUBPROBLEMS_FILTER)
].copy()
if "ml_callback_type" in cleaned_data_analysis.columns:
    exclude_callback_types = {"ML_at_integer_exact_at_LP"}
    cleaned_data_analysis = cleaned_data_analysis[
        ~cleaned_data_analysis["ml_callback_type"].isin(exclude_callback_types)
    ].copy()

cleaned_data_analysis_path = Path(OUTPUT_DIR) / "cleaned_data_analysis.xlsx"
with pd.ExcelWriter(cleaned_data_analysis_path, engine="openpyxl") as writer:
    cleaned_data_analysis.to_excel(writer, sheet_name="cleaned_data_analysis", index=False)
print(f"Exported cleaned_data_analysis: {cleaned_data_analysis_path}")

print(f"\nData splits prepared (after num_subproblems filter):") 
for group_name, group_df in analysis_results.items():
    print(f"  - {group_name}: {len(group_df)} rows")


GENERATING HIERARCHICAL ANALYSIS

Base output directory: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output

STOPPING-REASON ALIGNMENT DATA GROUP SPLIT
Exact OPTIMAL instances discovered: 20
all_optimal_baseline rows: 2357
not_all_optimal_baseline rows: 1847

APPLYING NUM_SUBPROBLEMS FILTER
Filtering to num_subproblems in [100, 300, 500]

Data groups to analyze: ['cleaned_data', 'exact_optimal_baseline', 'exact_nonoptimal_baseline', 'All_Optimal', 'NOT_all_Optimal']
ML methods kept for metric analysis: ['ML_Augmented_BD', 'Random_Sampling']
Callback types to analyze: ['total', 'ML_at_integer_exact_at_LP', 'ML_at_both', 'ML_at_LP_exact_at_integer', 'ML_at_both_and_ML_at_LP_exact_at_integer']
Exported cleaned_data analysis-filtered data: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/analysis_filtered/cleaned_data_analysis_filt

## Generate Tables and Figures

Write summary tables, line plots, and boxplots for every problem-class and callback-type folder.


## Analysis Helper Banner and Summary Table Writer


In [78]:
# =========================
# Analysis generation helper functions
# =========================
print("\n" + "="*60)
print("PREPARING DETAILED ANALYSIS HELPERS")
print("="*60)


def _write_metric_summary_tables(df_subset_report, tables_folder, filename_prefix=None):
    prefix = f"{filename_prefix}__" if filename_prefix else ""
    summary_specs = [
        (f"{prefix}summary_by_ml_method.csv", build_summary_by_ml_method),
        (f"{prefix}summary_by_ml_method_predictor.csv", build_summary_by_ml_method_and_predictor),
        (f"{prefix}summary_by_ml_method_covariates.csv", build_summary_by_ml_method_and_covariates),
        (f"{prefix}summary_by_ml_method_predictor_covariates.csv", build_summary_by_ml_method_predictor_covariates),
        (f"{prefix}summary_by_selected_subproblems_ratio_ml_method.csv", build_summary_by_selected_subproblems_ratio_ml_method),
        (f"{prefix}summary_by_selected_subproblems_ratio_ml_method_predictor.csv", build_summary_by_selected_subproblems_ratio_ml_method_and_predictor),
        (f"{prefix}summary_by_selected_subproblems_ratio_ml_method_covariates.csv", build_summary_by_selected_subproblems_ratio_ml_method_and_covariates),
        (f"{prefix}summary_by_selected_subproblems_ratio_ml_method_predictor_covariates.csv", build_summary_by_selected_subproblems_ratio_ml_method_predictor_covariates),
    ]
    for filename, builder in summary_specs:
        table = builder(df_subset_report)
        if not table.empty:
            table.to_csv(tables_folder / filename, index=False)
        del table
    gc.collect()



PREPARING DETAILED ANALYSIS HELPERS


## Analysis Line Plot Writer


In [79]:
def _write_metric_line_plots(df_subset_report, line_plots_folder, filter_dict, filename_prefix=None):
    prefix = f"{filename_prefix}__" if filename_prefix else ""
    line_plot_specs = [
        (plot_metric_vs_num_subproblems_by_ml_method, "Optimality_Gap%", f"{prefix}gap_vs_num_subproblems_by_ml_method.png"),
        (plot_metric_vs_num_subproblems_by_ml_method, "Time_Ratio", f"{prefix}time_ratio_vs_num_subproblems_by_ml_method.png"),
        (plot_metric_vs_num_subproblems_by_predictor, "Optimality_Gap%", f"{prefix}gap_vs_num_subproblems_by_predictor.png"),
        (plot_metric_vs_num_subproblems_by_predictor, "Time_Ratio", f"{prefix}time_ratio_vs_num_subproblems_by_predictor.png"),
        (plot_metric_vs_num_subproblems_by_covariates, "Optimality_Gap%", f"{prefix}gap_vs_num_subproblems_by_covariates.png"),
        (plot_metric_vs_num_subproblems_by_covariates, "Time_Ratio", f"{prefix}time_ratio_vs_num_subproblems_by_covariates.png"),
        (plot_metric_vs_num_subproblems_by_predictor_covariates, "Optimality_Gap%", f"{prefix}gap_vs_num_subproblems_by_predictor_covariates.png"),
        (plot_metric_vs_num_subproblems_by_predictor_covariates, "Time_Ratio", f"{prefix}time_ratio_vs_num_subproblems_by_predictor_covariates.png"),
        (plot_metric_vs_selected_subproblems_ratio_by_ml_method, "Optimality_Gap%", f"{prefix}gap_vs_selected_subproblems_ratio_by_ml_method.png"),
        (plot_metric_vs_selected_subproblems_ratio_by_ml_method, "Time_Ratio", f"{prefix}time_ratio_vs_selected_subproblems_ratio_by_ml_method.png"),
        (plot_metric_vs_selected_subproblems_ratio_by_predictor, "Optimality_Gap%", f"{prefix}gap_vs_selected_subproblems_ratio_by_predictor.png"),
        (plot_metric_vs_selected_subproblems_ratio_by_predictor, "Time_Ratio", f"{prefix}time_ratio_vs_selected_subproblems_ratio_by_predictor.png"),
        (plot_metric_vs_selected_subproblems_ratio_by_covariates, "Optimality_Gap%", f"{prefix}gap_vs_selected_subproblems_ratio_by_covariates.png"),
        (plot_metric_vs_selected_subproblems_ratio_by_covariates, "Time_Ratio", f"{prefix}time_ratio_vs_selected_subproblems_ratio_by_covariates.png"),
        (plot_metric_vs_selected_subproblems_ratio_by_predictor_covariates, "Optimality_Gap%", f"{prefix}gap_vs_selected_subproblems_ratio_by_predictor_covariates.png"),
        (plot_metric_vs_selected_subproblems_ratio_by_predictor_covariates, "Time_Ratio", f"{prefix}time_ratio_vs_selected_subproblems_ratio_by_predictor_covariates.png"),
    ]
    for plotter, metric, filename in line_plot_specs:
        plotter(df_subset_report, metric=metric, output_path=line_plots_folder / filename, filter_dict=filter_dict)
        plt.close("all")
    gc.collect()


## Analysis Boxplot Writer


In [80]:
def _write_metric_boxplots(df_subset_report, boxplots_folder, filter_dict, filename_prefix=None):
    prefix = f"{filename_prefix}__" if filename_prefix else ""
    boxplot_specs = [
        (plot_boxplots_by_ml_method, "Optimality_Gap%", f"{prefix}boxplot_gap_by_ml_method.png", {}),
        (plot_boxplots_by_ml_method, "Time_Ratio", f"{prefix}boxplot_time_ratio_by_ml_method.png", {}),
        (plot_boxplots_by_predictor, "Optimality_Gap%", f"{prefix}boxplot_gap_by_predictor.png", {}),
        (plot_boxplots_by_predictor, "Time_Ratio", f"{prefix}boxplot_time_ratio_by_predictor.png", {}),
        (plot_boxplots_by_covariates, "Optimality_Gap%", f"{prefix}boxplot_gap_by_covariates.png", {}),
        (plot_boxplots_by_covariates, "Time_Ratio", f"{prefix}boxplot_time_ratio_by_covariates.png", {}),
        (plot_boxplots_by_ml_method, "Optimality_Gap%", f"{prefix}boxplot_gap_by_ml_method_selected_subproblems_ratio.png", {"split_column": "selected_subproblems_ratio", "split_prefix": "ratio"}),
        (plot_boxplots_by_ml_method, "Time_Ratio", f"{prefix}boxplot_time_ratio_by_ml_method_selected_subproblems_ratio.png", {"split_column": "selected_subproblems_ratio", "split_prefix": "ratio"}),
        (plot_boxplots_by_predictor, "Optimality_Gap%", f"{prefix}boxplot_gap_by_predictor_selected_subproblems_ratio.png", {"split_column": "selected_subproblems_ratio", "split_prefix": "ratio"}),
        (plot_boxplots_by_predictor, "Time_Ratio", f"{prefix}boxplot_time_ratio_by_predictor_selected_subproblems_ratio.png", {"split_column": "selected_subproblems_ratio", "split_prefix": "ratio"}),
        (plot_boxplots_by_covariates, "Optimality_Gap%", f"{prefix}boxplot_gap_by_covariates_selected_subproblems_ratio.png", {"split_column": "selected_subproblems_ratio", "split_prefix": "ratio"}),
        (plot_boxplots_by_covariates, "Time_Ratio", f"{prefix}boxplot_time_ratio_by_covariates_selected_subproblems_ratio.png", {"split_column": "selected_subproblems_ratio", "split_prefix": "ratio"}),
    ]
    for plotter, metric, filename, extra_kwargs in boxplot_specs:
        plotter(df_subset_report, metric=metric, output_path=boxplots_folder / filename, filter_dict=filter_dict, **extra_kwargs)
        plt.close("all")
    gc.collect()


## Analysis Callback and Class Helpers


In [81]:
def _filter_callback_subset(df_class, callback_type):
    if callback_type == "total":
        return df_class.copy()
    callback_values = COMBINED_CALLBACK_TYPE_MAP.get(callback_type, [callback_type])
    df_ml_callback = df_class[(df_class["method_category"] == "ML") & (df_class["ml_callback_type"].isin(callback_values))]
    df_exact = df_class[df_class["method_category"] == "Exact"]
    return pd.concat([df_ml_callback, df_exact], ignore_index=True)


def _analysis_filter_dict(callback_type):
    if callback_type == "total" or callback_type in COMBINED_CALLBACK_TYPE_MAP:
        return {}
    return {"ml_callback_type": callback_type}


def _analysis_problem_classes(group_df):
    return ["total"] + sorted([x for x in group_df["problem_class"].unique().tolist() if pd.notna(x)])


## Analysis Generation: Callback Writer


In [82]:
def generate_callback_analysis(df_class, callback_type, callback_folder):
    df_status_subset = _filter_callback_subset(df_class, callback_type)
    df_metric_subset = filter_to_analysis_methods(df_status_subset)
    tables_folder = callback_folder / "tables"
    line_plots_folder = callback_folder / "figures" / "line_plots"
    boxplots_folder = callback_folder / "figures" / "boxplots"
    tables_folder.mkdir(parents=True, exist_ok=True)
    line_plots_folder.mkdir(parents=True, exist_ok=True)
    boxplots_folder.mkdir(parents=True, exist_ok=True)
    print(f"      Records for status table: {len(df_status_subset)}")
    print(f"      Records for metric analysis after ML-method filter: {len(df_metric_subset)}")

    relative_parts = callback_folder.relative_to(Path(OUTPUT_DIR)).parts
    folder_tag = "_".join(relative_parts[:3]) if relative_parts else None

    input_data_name = (
        f"{folder_tag}__analysis_input_data.csv"
        if folder_tag
        else "analysis_input_data.csv"
    )
    df_status_subset.to_csv(tables_folder / input_data_name, index=False)

    nonoptimal_table = build_nonoptimal_percentage_by_instance_table(df_status_subset)
    nonoptimal_name = (
        f"{folder_tag}__nonoptimal_percentage_by_instance_num_subproblems.csv"
        if folder_tag
        else "nonoptimal_percentage_by_instance_num_subproblems.csv"
    )
    nonoptimal_table.to_csv(tables_folder / nonoptimal_name, index=False)
    del nonoptimal_table

    method_summary = build_method_optimality_summary(df_status_subset)
    method_by_instance = build_method_optimality_by_instance(df_status_subset)
    method_summary_name = (
        f"{folder_tag}__method_optimality_summary.xlsx"
        if folder_tag
        else "method_optimality_summary.xlsx"
    )
    method_summary_path = tables_folder / method_summary_name
    with pd.ExcelWriter(method_summary_path, engine="openpyxl") as writer:
        method_summary.to_excel(writer, sheet_name="overall", index=False)
        method_by_instance.to_excel(writer, sheet_name="by_instance", index=False)
    del method_summary, method_by_instance

    df_subset_report = apply_presentation_naming(df_metric_subset)
    filter_dict = _analysis_filter_dict(callback_type)
    _write_metric_summary_tables(df_subset_report, tables_folder, filename_prefix=folder_tag)
    _write_metric_line_plots(df_subset_report, line_plots_folder, filter_dict, filename_prefix=folder_tag)
    _write_metric_boxplots(df_subset_report, boxplots_folder, filter_dict, filename_prefix=folder_tag)
    del df_status_subset, df_metric_subset, df_subset_report
    plt.close("all")
    gc.collect()


## Analysis Generation: Data Group Writer


In [83]:
def generate_analysis_for_data_group(data_group_name, group_df):
    group_folder = Path(OUTPUT_DIR) / data_group_name
    group_folder.mkdir(parents=True, exist_ok=True)
    problem_classes = _analysis_problem_classes(group_df)
    total_iterations = len(problem_classes) * len(callback_types)
    current_iteration = 0
    print(f"\n{'='*50}")
    print(f"Analyzing data_group: {data_group_name}")
    print(f"Records in data group: {len(group_df)}")
    print(f"Problem classes: {problem_classes}")
    print(f"{'='*50}")
    for problem_class in problem_classes:
        if problem_class == "total":
            df_class = group_df.copy()
            class_folder = group_folder / "total"
        else:
            df_class = group_df[group_df["problem_class"] == problem_class].copy()
            class_folder = group_folder / str(problem_class)
        class_folder.mkdir(parents=True, exist_ok=True)
        print(f"\n  Problem class: {problem_class} ({len(df_class)} records)")
        for callback_type in callback_types:
            current_iteration += 1
            print(f"\n    [{current_iteration}/{total_iterations}] Callback type: {callback_type}")
            generate_callback_analysis(df_class, callback_type, class_folder / callback_type)
        del df_class
        gc.collect()
    print(f"\nCompleted data_group: {data_group_name}")


## Generate Analysis for Cleaned Data


In [84]:
generate_analysis_for_data_group("cleaned_data", analysis_data_groups["cleaned_data"])



Analyzing data_group: cleaned_data
Records in data group: 3663
Problem classes: ['total', 'SCFLP', 'SCMND']

  Problem class: total (3663 records)

    [1/15] Callback type: total
      Records for status table: 3663
      Records for metric analysis after ML-method filter: 2166
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/cleaned_data/total/total/figures/line_plots/cleaned_data_total_total__gap_vs_num_subproblems_by_ml_method.png
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/cleaned_data/total/total/figures/line_plots/cleaned_data_total_total__time_ratio_vs_num_subproblems_by_ml_method.png
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/cleaned_data/total/total/figures/line_plots/cleaned_data_total_total__time_ratio_

## Generate Analysis for Exact OPTIMAL Baseline Group


In [85]:
generate_analysis_for_data_group("exact_optimal_baseline", analysis_data_groups["exact_optimal_baseline"])



Analyzing data_group: exact_optimal_baseline
Records in data group: 1933
Problem classes: ['total', 'SCFLP', 'SCMND']

  Problem class: total (1933 records)

    [1/15] Callback type: total
      Records for status table: 1933
      Records for metric analysis after ML-method filter: 1024
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/exact_optimal_baseline/total/total/figures/line_plots/exact_optimal_baseline_total_total__gap_vs_num_subproblems_by_ml_method.png
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/exact_optimal_baseline/total/total/figures/line_plots/exact_optimal_baseline_total_total__time_ratio_vs_num_subproblems_by_ml_method.png
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/exact_optimal_baseline/total/to

## Generate Analysis for Exact Non-OPTIMAL Baseline Group


In [86]:
generate_analysis_for_data_group("exact_nonoptimal_baseline", analysis_data_groups["exact_nonoptimal_baseline"])

print("\n" + "="*60)
print("ANALYSIS GENERATION COMPLETE")
print("="*60)


Analyzing data_group: exact_nonoptimal_baseline
Records in data group: 1730
Problem classes: ['total', 'SCFLP', 'SCMND']

  Problem class: total (1730 records)

    [1/15] Callback type: total
      Records for status table: 1730
      Records for metric analysis after ML-method filter: 1142
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/exact_nonoptimal_baseline/total/total/figures/line_plots/exact_nonoptimal_baseline_total_total__gap_vs_num_subproblems_by_ml_method.png
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/exact_nonoptimal_baseline/total/total/figures/line_plots/exact_nonoptimal_baseline_total_total__time_ratio_vs_num_subproblems_by_ml_method.png
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/exact_nonoptimal

## Generate Analysis for All_Optimal Group



In [87]:
generate_analysis_for_data_group("All_Optimal", analysis_data_groups["All_Optimal"])


Analyzing data_group: All_Optimal
Records in data group: 1933
Problem classes: ['total', 'SCFLP', 'SCMND']

  Problem class: total (1933 records)

    [1/15] Callback type: total
      Records for status table: 1933
      Records for metric analysis after ML-method filter: 1024
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/All_Optimal/total/total/figures/line_plots/All_Optimal_total_total__gap_vs_num_subproblems_by_ml_method.png
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/All_Optimal/total/total/figures/line_plots/All_Optimal_total_total__time_ratio_vs_num_subproblems_by_ml_method.png
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/All_Optimal/total/total/figures/line_plots/All_Optimal_total_total__time_ratio_vs_num_

## Generate Analysis for NOT_all_Optimal Group



In [88]:
generate_analysis_for_data_group("NOT_all_Optimal", analysis_data_groups["NOT_all_Optimal"])

print("\n" + "="*60)
print("ALL DATA GROUP ANALYSIS GENERATION COMPLETE")
print("="*60)


Analyzing data_group: NOT_all_Optimal
Records in data group: 1730
Problem classes: ['total', 'SCFLP', 'SCMND']

  Problem class: total (1730 records)

    [1/15] Callback type: total
      Records for status table: 1730
      Records for metric analysis after ML-method filter: 1142
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/NOT_all_Optimal/total/total/figures/line_plots/NOT_all_Optimal_total_total__gap_vs_num_subproblems_by_ml_method.png
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/NOT_all_Optimal/total/total/figures/line_plots/NOT_all_Optimal_total_total__time_ratio_vs_num_subproblems_by_ml_method.png
Saved: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/NOT_all_Optimal/total/total/figures/line_plots/NOT_all_Optimal_tot

## Export Workbook

Save the processed datasets and audit sheets to the Excel workbook.


In [89]:

# =========================
# STEP 8: Export Excel workbook and group CSV files
# =========================
print("\n" + "="*60)
print("EXPORTING CLEANED GROUP FILES AND EXCEL WORKBOOK")
print("="*60)

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

for group_name, group_df in analysis_results.items():
    csv_path = output_dir / f"{group_name}.csv"
    group_df.to_csv(csv_path, index=False)
    print(f"Exported {group_name}.csv with {len(group_df)} rows")

output_excel_path = output_dir / OUTPUT_EXCEL_NAME

with pd.ExcelWriter(output_excel_path, engine="openpyxl") as writer:
    for group_name, group_df in analysis_results.items():
        group_df.to_excel(writer, sheet_name=group_name[:31], index=False)
        print(f"Exported '{group_name[:31]}' tab with {len(group_df)} rows")

    if not removed_duplicates.empty:
        removed_duplicates.to_excel(writer, sheet_name="removed_duplicates", index=False)
        print(f"Exported 'removed_duplicates' tab with {len(removed_duplicates)} rows")

print(f"\nExcel file saved to: {output_excel_path}")



EXPORTING CLEANED GROUP FILES AND EXCEL WORKBOOK
Exported cleaned_data.csv with 3663 rows
Exported exact_optimal_baseline.csv with 1933 rows
Exported exact_nonoptimal_baseline.csv with 1730 rows
Exported All_Optimal.csv with 1933 rows
Exported NOT_all_Optimal.csv with 1730 rows
Exported 'cleaned_data' tab with 3663 rows
Exported 'exact_optimal_baseline' tab with 1933 rows
Exported 'exact_nonoptimal_baseline' tab with 1730 rows
Exported 'All_Optimal' tab with 1933 rows
Exported 'NOT_all_Optimal' tab with 1730 rows
Exported 'removed_duplicates' tab with 4543 rows

Excel file saved to: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/processed_experiment_analysis.xlsx


## Export Coverage Counts Workbook

Write two pivot-style count tables to a separate Excel file to spot missing experiments.

In [90]:
# =========================
# STEP 8B: Export experiment coverage counts workbook
# =========================
print("\n" + "=" * 60)
print("EXPORTING EXPERIMENT COVERAGE COUNTS")
print("=" * 60)

coverage_source = df_cleaned_data.copy()
coverage_source = coverage_source.dropna(subset=["determisitic_instance_name", "num_subproblems"]).copy()

tab1_index = ["determisitic_instance_name", "num_subproblems"]
tab1_columns = ["method_category", "ml_method_type", "ml_callback_type"]
tab2_columns = [
    "method_category",
    "ml_method_type",
    "ml_callback_type",
    "predictor_type",
    "selected_subproblems_ratio",
    "use_covariates",
]

tab1_counts = (
    coverage_source.pivot_table(
        index=tab1_index,
        columns=tab1_columns,
        values="instance_filename",
        aggfunc="size",
        fill_value=0,
    )
    .sort_index()
)
if tab1_counts.columns.nlevels == len(tab1_columns):
    tab1_counts.columns = tab1_counts.columns.set_names(tab1_columns)
    tab1_counts = tab1_counts.reorder_levels(tab1_columns, axis=1).sort_index(axis=1)
elif len(tab1_counts.columns) == 0:
    tab1_counts.columns = pd.MultiIndex.from_arrays([[]] * len(tab1_columns), names=tab1_columns)

tab2_counts = (
    coverage_source.pivot_table(
        index=tab1_index,
        columns=tab2_columns,
        values="instance_filename",
        aggfunc="size",
        fill_value=0,
    )
    .sort_index()
)
if tab2_counts.columns.nlevels == len(tab2_columns):
    tab2_counts.columns = tab2_counts.columns.set_names(tab2_columns)
    tab2_counts = tab2_counts.reorder_levels(tab2_columns, axis=1).sort_index(axis=1)
elif len(tab2_counts.columns) == 0:
    tab2_counts.columns = pd.MultiIndex.from_arrays([[]] * len(tab2_columns), names=tab2_columns)

coverage_excel_path = Path(OUTPUT_DIR) / OUTPUT_EXPERIMENT_COVERAGE_EXCEL_NAME
with pd.ExcelWriter(coverage_excel_path, engine="openpyxl") as writer:
    tab1_counts.to_excel(writer, sheet_name="method_callback_counts")
    tab2_counts.to_excel(writer, sheet_name="full_factor_counts")

print(f"Coverage workbook saved to: {coverage_excel_path}")


EXPORTING EXPERIMENT COVERAGE COUNTS
Coverage workbook saved to: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output/experiment_coverage_counts.xlsx


## Final Summary

Print final row counts, coverage statistics, and the generated output structure.


In [91]:


# =========================
# STEP 9: Summary statistics
# =========================
print("\n" + "="*60)
print("FINAL SUMMARY STATISTICS")
print("="*60)

print(f"\nDataset split summary:")
for group_name, group_df in analysis_results.items():
    print(f"  {group_name}: {len(group_df)}")
    print(f"    - Exact methods: {len(group_df[group_df['method_category'] == 'Exact'])}")
    print(f"    - ML methods: {len(group_df[group_df['method_category'] == 'ML'])}")

print(f"\nAnalysis dataset filter:")
print(f"  Exact rows are retained")
print(f"  ML methods retained: {ANALYSIS_ML_METHODS}")

ml_rows = df_exact_optimal_baseline[df_exact_optimal_baseline["method_category"] == "ML"]
ml_with_gap = ml_rows[ml_rows["Optimality_Gap%"].notna()]
print(f"\nExact OPTIMAL baseline group metric coverage:")
print(f"  Total ML rows: {len(ml_rows)}")
print(f"  ML rows with Optimality_Gap%: {len(ml_with_gap)}")
print(f"  Coverage: {100*len(ml_with_gap)/max(len(ml_rows), 1):.1f}%")

print(f"\nProblem class distribution (exact_optimal_baseline):")
print(df_exact_optimal_baseline["problem_class"].value_counts())

print(f"\nMethod distribution (exact_optimal_baseline, ML only):")
print(df_exact_optimal_baseline[df_exact_optimal_baseline["method_category"] == "ML"]["ml_method_type"].value_counts())

print(f"\nCallback type distribution (exact_optimal_baseline, ML only):")
print(df_exact_optimal_baseline[df_exact_optimal_baseline["method_category"] == "ML"]["ml_callback_type"].value_counts())

print(f"\nOutput directory structure with 5 data groups:")
print(f"  {OUTPUT_DIR}")
print(f"  |-- {OUTPUT_EXCEL_NAME}")
print(f"  |-- {OUTPUT_DEDUPED_CSV_NAME}")
print(f"  |-- cleaned_data.csv")
print(f"  |-- exact_optimal_baseline.csv")
print(f"  |-- exact_nonoptimal_baseline.csv")
print(f"  |-- All_Optimal.csv")
print(f"  |-- NOT_all_Optimal.csv")
print(f"  |-- cleaned_data_analysis.xlsx")
print(f"  |-- cleaned_data/ (problem_class subdirs: total, SCFLP, SCMND, ...)")
print(f"  |-- exact_optimal_baseline/")
print(f"  |-- exact_nonoptimal_baseline/")
print(f"  |-- All_Optimal/")
print(f"  |   |-- total/")
print(f"  |   |   `-- total/")
print(f"  |   |       |-- tables/")
print(f"  |   |       |   |-- nonoptimal_percentage_by_instance_num_subproblems.csv")
print(f"  |   |       |   |-- summary_by_ml_method.csv")
print(f"  |   |       |   `-- ... (other summary tables)")
print(f"  |   |       `-- figures/")
print(f"  |   |           |-- line_plots/ (gap and time_ratio plots)")
print(f"  |   |           `-- boxplots/")
print(f"  |   `-- [problem_class]/ (SCFLP, SCMND, etc. - same structure)")
print(f"  `-- NOT_all_Optimal/ (same structure as All_Optimal)")



FINAL SUMMARY STATISTICS

Dataset split summary:
  cleaned_data: 3663
    - Exact methods: 38
    - ML methods: 3625
  exact_optimal_baseline: 1933
    - Exact methods: 16
    - ML methods: 1917
  exact_nonoptimal_baseline: 1730
    - Exact methods: 22
    - ML methods: 1708
  All_Optimal: 1933
    - Exact methods: 16
    - ML methods: 1917
  NOT_all_Optimal: 1730
    - Exact methods: 22
    - ML methods: 1708

Analysis dataset filter:
  Exact rows are retained
  ML methods retained: ['ML_Augmented_BD', 'Random_Sampling']

Exact OPTIMAL baseline group metric coverage:
  Total ML rows: 2337
  ML rows with Optimality_Gap%: 2335
  Coverage: 99.9%

Problem class distribution (exact_optimal_baseline):
problem_class
SCFLP    2123
SCMND     234
Name: count, dtype: int64

Method distribution (exact_optimal_baseline, ML only):
ml_method_type
ML_Augmented_BD               1525
ML_Augmented_BD_with_duals     318
Uncertainty_Similarity         314
Random_Sampling                180
Name: count, d

## Completion Message

List the main output files and artifact families created by the workflow.


In [92]:
# =========================
# COMPLETED ANALYSIS WORKFLOW
# =========================
print("\n" + "="*60)
print("ANALYSIS WORKFLOW COMPLETE")
print("="*60)
print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(f"\nMain files:")
print(f"  - {OUTPUT_EXCEL_NAME}: Main workbook with 5 data-group tabs (cleaned_data, exact_optimal_baseline, exact_nonoptimal_baseline, All_Optimal, NOT_all_Optimal) and audit tabs when available")
print(f"  - {OUTPUT_DEDUPED_CSV_NAME}: Deduplicated dataset saved immediately after duplicate removal")
print(f"  - {OUTPUT_EXPERIMENT_COVERAGE_EXCEL_NAME}: Coverage counts workbook for missing experiment checks")
print(f"  - cleaned_data_analysis.xlsx: Cleaned data after method and num_subproblems filters")
print(f"  - cleaned_data.csv, exact_optimal_baseline.csv, exact_nonoptimal_baseline.csv, All_Optimal.csv, NOT_all_Optimal.csv")
print(f"\nData Groups and Filtering:")
print(f"  1. cleaned_data: All deduplicated rows")
print(f"  2. exact_optimal_baseline: Rows from instances with Exact OPTIMAL baseline")
print(f"  3. exact_nonoptimal_baseline: Rows from instances without Exact OPTIMAL baseline")
print(f"  4. All_Optimal: Rows where BOTH Exact and ML methods achieved OPTIMAL (filtered by num_subproblems)")
print(f"  5. NOT_all_Optimal: Rows where at least one method did not achieve OPTIMAL (filtered by num_subproblems)")
print(f"\nNum_Subproblems Filter Applied: {NUM_SUBPROBLEMS_FILTER} only")
print(f"\nFolder structure for each data group (All_Optimal, NOT_all_Optimal shown below):")
print(f"  - Data_Group/")
print(f"    ├── total/")
print(f"    │   └── total/")
print(f"    │       ├── tables/")
print(f"    │       │   ├── nonoptimal_percentage_by_instance_num_subproblems.csv")
print(f"    │       │   ├── summary_by_ml_method.csv")
print(f"    │       │   ├── summary_by_ml_method_predictor.csv")
print(f"    │       │   ├── summary_by_ml_method_covariates.csv")
print(f"    │       │   ├── summary_by_selected_subproblems_ratio_ml_method.csv")
print(f"    │       │   ├── summary_by_selected_subproblems_ratio_ml_method_predictor.csv")
print(f"    │       │   └── summary_by_selected_subproblems_ratio_ml_method_covariates.csv")
print(f"    │       └── figures/")
print(f"    │           ├── line_plots/")
print(f"    │           │   ├── gap_vs_num_subproblems_by_ml_method.png")
print(f"    │           │   ├── time_ratio_vs_num_subproblems_by_ml_method.png")
print(f"    │           │   ├── gap_vs_selected_subproblems_ratio_by_ml_method.png")
print(f"    │           │   ├── time_ratio_vs_selected_subproblems_ratio_by_ml_method.png")
print(f"    │           │   └── ... (other plots with predictor/covariate combinations)")
print(f"    │           └── boxplots/ (metric distributions)")
print(f"    └── [PROBLEM_CLASS]/ (e.g., SCFLP, SCMND)")
print(f"        └── [same folder structure as total/]")
print(f"\nML Methods Included in Analysis: {ANALYSIS_ML_METHODS}")
print(f"Time_Ratio Reference Line: y=1 (dashed) indicates where ML time equals Exact time")



ANALYSIS WORKFLOW COMPLETE

All outputs saved to: /mnt/c/Programming/Research Projects/Benders_Integration_with_ML/RESULTS/Experiments - Digital Research Alliance of Canada/Output

Main files:
  - processed_experiment_analysis.xlsx: Main workbook with 5 data-group tabs (cleaned_data, exact_optimal_baseline, exact_nonoptimal_baseline, All_Optimal, NOT_all_Optimal) and audit tabs when available
  - deduplicated_experiment_data.csv: Deduplicated dataset saved immediately after duplicate removal
  - experiment_coverage_counts.xlsx: Coverage counts workbook for missing experiment checks
  - cleaned_data_analysis.xlsx: Cleaned data after method and num_subproblems filters
  - cleaned_data.csv, exact_optimal_baseline.csv, exact_nonoptimal_baseline.csv, All_Optimal.csv, NOT_all_Optimal.csv

Data Groups and Filtering:
  1. cleaned_data: All deduplicated rows
  2. exact_optimal_baseline: Rows from instances with Exact OPTIMAL baseline
  3. exact_nonoptimal_baseline: Rows from instances without 